In [ ]:
from google.colab import drive
drive.mount('/content/mydrive')

Mounted at /content/mydrive


#!/usr/bin/env python3
# ============================================================
# RECLASSIFY ALL — TUMC Dataset Keyword-Based Classification
# ============================================================
# Applies URL keyword-based classification to dataset_with_folds_v10.csv
# using Turkish threat lexicons and source confidence logic.
#
# Usage:
#   python reclassify_dataset.py
#
# Input:  dataset_with_folds_v10.csv
# Output: dataset_with_folds_v11.csv
# ============================================================

In [ ]:
#!/usr/bin/env python3
# ============================================================
# RECLASSIFY ALL — TurkMAL Dataset Keyword-Based Classification
# ============================================================
# Applies URL keyword-based classification to dataset_with_folds_v10.csv
# using Turkish threat lexicons and source confidence logic.
#
# Usage:
#   python reclassify_dataset.py
#
# Input:  dataset_with_folds_v10.csv
# Output: dataset_with_folds_v10_reclassified.csv
# ============================================================

import re
import pandas as pd
from pathlib import Path

# ════════════════════════════════════════════════════════════
# SOURCE CONFIDENCE CONFIG
# ════════════════════════════════════════════════════════════
SOURCE_CONFIG = {
    "PhishTank":  ("phishing", 0.95),
    "URLhaus":    ("malware",  0.90),
    "OpenPhish":  ("phishing", 0.85),
    "ThreatFox":  ("malware",  0.92),
    "USOM":       (None,       0.70),
    "usom":       (None,       0.70),
    "phishstats": ("phishing", 0.80),
    "otx":        (None,       0.75),
    "cinsscore":  ("malware",  0.70),
    "blocklist":  ("malware",  0.72),
    "__default__": (None,      0.65),
}

RECLASSIFY_ALL_SOURCES = {"USOM", "usom", "otx"}
KEEP_SOURCE_LABEL = {"PhishTank", "URLhaus", "OpenPhish", "ThreatFox"}

# ════════════════════════════════════════════════════════════
# PHISHING KEYWORDS — 270+
# ════════════════════════════════════════════════════════════
PHISHING_KEYWORDS = [
    # Turkish authentication / account
    "giris","giriş","oturum","uye-giris","uyegiris",
    "giris-yap","girisyap","hesabim","hesabım",
    "hesap-dogrula","sifre","şifre","sifre-sifirla",
    "sifre-yenile","parola","kullanici","kullanıcı",
    "uyelik","üyelik",
    # Turkish payment / banking
    "odeme","ödeme","banka","bankaci","bankacilik","bankacılık",
    "mobil-banka","internet-bankaciligi","kart-bilgi",
    "kart-dogrulama","iban","swift","havale","eft",
    "para-transferi","para-gonder","kredi-karti","kredikarti",
    "debit-karti","banka-hesabi","hesap-bakiye",
    # Turkish government
    "e-devlet","edevlet","e-nabiz","enabiz","e-okul",
    "mhrs","sgk","vergi-dairesi","gib","nvi",
    "tckimlik","tc-kimlik","tc-no","tc-numarasi",
    "kimlik-dogrulama","pasaport","ehliyet",
    # Turkish logistics
    "kargo-takip","kargom","teslimat","gonderitakibi",
    "siparis","sipariş","siparis-takip","fatura",
    "aras-kargo","araskargo","yurtici-kargo",
    "mng-kargo","surat-kargo","ups-kargo",
    # Turkish verification
    "dogrula","doğrula","dogrulama","doğrulama",
    "kimlik-dogrula","guvenli","güvenli",
    "guvenlik","güvenlik","guvenlik-kodu",
    "onay","onayla","onaylama",
    "sms-dogrulama","otp-dogrulama",
    # Turkish bank brands
    "garanti-bbva","garantibbva","garanti-bankasi",
    "akbank","isbank","isbankasi","is-bankasi",
    "ziraat-bankasi","ziraatbankasi",
    "halk-bank","halkbank","vakif-bank","vakifbank",
    "yapi-kredi","yapikredi","deniz-bank","denizbank",
    "qnb-finansbank","teb-bankasi","enpara",
    "papara","paycell","tosla",
    "paribu","btcturk","binance-tr",
    # Turkish e-commerce / delivery brands
    "trendyol","hepsiburada","n11",
    "ciceksepeti","gittigidiyor","sahibinden",
    # Turkish telecom brands
    "turkcell","vodafone-tr","turktelekom","ttnet","bimcell",
    # Crypto wallet phishing
    "metamask","metamask-login","metamask-signin",
    "metamask-wallet","metamask-connect",
    "wallet-connect","walletconnect",
    "trustwallet","trust-wallet",
    "coinbase-wallet","coinbase-login",
    "uniswap-login","opensea-login",
    # Generic phishing (English)
    "login","log-in","signin","sign-in","sign-on",
    "account-verify","account-update","account-suspended",
    "account-locked","account-confirm",
    "secure-login","securelogin","banking-login","banklogin",
    "my-account","myaccount","verify-account",
    "confirm-account","update-account",
    "credential","credentials","webmail","webbanking",
    "online-banking","customer-portal","client-portal",
    # Credential harvesting
    "password-reset","reset-password","forgot-password",
    "change-password","unlock-account","reactivate",
    "recover-account","account-recovery",
    "two-factor","2fa","one-time-password",
    # Global brands
    "paypal","microsoft-login","microsoftlogin",
    "apple-id","appleid","icloud-login",
    "google-login","gmail-login","facebook-login",
    "instagram-login","netflix-login","amazon-login",
    "ebay-login","dhl-tracking","fedex-tracking","ups-tracking",
    # URL structural signals
    "phish","-login\\.","musteri-hizmetleri","musteri-destek",
    "destek-hatti","islem","musteri",
    "basvuru","kampanya","hesap-giris","kredi","iade",
    # Sahibinden / classified ad phishing
    "sahibinden-giris","letgo-giris","emlak-giris",
    # Hediye / gift phishing
    "hediye-kazan","hediyekampanya","hediyeniz",
]

# ════════════════════════════════════════════════════════════
# MALWARE KEYWORDS — 130+
# ════════════════════════════════════════════════════════════
MALWARE_KEYWORDS = [
    # Executable extensions
    r"\.exe", r"\.msi", r"\.dll", r"\.bat", r"\.cmd",
    r"\.ps1", r"\.vbs", r"\.hta", r"\.scr", r"\.pif",
    r"\.application",
    # Archives
    r"\.zip(?=[/?#]|$)", r"\.rar(?=[/?#]|$)",
    r"\.7z(?=[/?#]|$)", r"\.tar(?=[/?#]|$)",
    r"\.gz(?=[/?#]|$)", r"\.iso(?=[/?#]|$)",
    r"\.img(?=[/?#]|$)",
    # Office macro documents
    r"\.docm", r"\.xlsm", r"\.xlam", r"\.pptm",
    # Mobile malware
    r"\.apk", r"\.apks", r"\.xapk", r"\.ipa",
    # Scripts and binaries
    r"\.sh(?=[/?#]|$)", r"\.jar(?=[/?#]|$)",
    r"\.pfm", r"\.bin(?=[/?#]|$)", r"\.dat(?=[/?#]|$)",
    r"\.chk(?=[/?#]|$)", r"\.filter(?=[/?#]|$)",
    # C2 / botnet
    "botnet","c2server","c2-server","command-and-control",
    "cnc-server","rat-panel","panel/gate","gate\\.php",
    "bot\\.php","check-in","checkin","beacon","callback",
    "exfil","webshell","shell\\.php","c99\\.php","r57\\.php",
    # Malware families
    "malware","ransomware","trojan","worm","rootkit",
    "spyware","adware","keylogger","stealware","infostealer",
    "stealer","miner","cryptominer","cryptojack","backdoor",
    "exploit","payload","dropper","loader","shellcode",
    "cobalt-strike","metasploit","meterpreter","mimikatz",
    "emotet","trickbot","qakbot","formbook","redline",
    "raccoon","agent-tesla","asyncrat","remcos","nanocore",
    "njrat","darkcomet","quasar","lokibot","azorult",
    "amadey","vidar","mars-stealer","orcus","xworm",
    "dcrat","warzone","venomrat","sliver","havoc",
    # Delivery patterns
    "download\\.php","get-file","getfile","fetch-payload",
    "malware-download","file-download","soft-download",
    "crack-download","keygen-download",
    # Document malware
    "invoice\\.docm","fatura\\.xlsm","order\\.docm",
    "payment\\.xlsm","receipt\\.xlsm",
    # Mobile malware
    "hack-apk","mod-apk","cheat-apk","premium-apk",
    "crack-apk","modded-apk","unlimited-apk",
    # Reverse shells
    "reverse-shell","bind-shell","nc-shell","bash-shell",
    # Turkish malware
    "virüs","truva","fidye","zararlı","kötücül",
    "hileli-apk","crack-indir","full-program",
    "aktivator","keygen","seri-no","lisans-crack",
]

# ════════════════════════════════════════════════════════════
# SCAM KEYWORDS — 150+
# ════════════════════════════════════════════════════════════
SCAM_KEYWORDS = [
    # Turkish prize / giveaway
    "kazandiniz","kazandınız","odul","ödül","hediye",
    "cekilis","çekiliş","bedava","ucretsiz","ücretsiz",
    "kupon","ikramiye","hibe","bedava-iphone",
    # Turkish investment / crypto
    "yatirim","yatırım","kripto","bitcoin","kazanc","kazanç",
    "borsa","forex","dogalgaz","doğalgaz",
    "parakazan","ekgelir","evdecalis","evdeçalış",
    "airdrop","gorev-yap","görev-yap",
    "forex-robotu","yapayzeka-kazanc","ai-yatirim",
    # Turkish gambling
    "bahis","kumar","casino","iddaa","jackpot","slot","rulet",
    "sweetbonanza","denemebonusu","poker","blackjack",
    "canli-casino","canlı-casino","canlibahis",
    "kayip-bonusu","freebet","freespin","bahissiteleri",
    # Turkish charity fraud
    "bagis","bağış","afaddestek","kizilaydestek",
    "deprem","yardim-vakfi","zekat-bagis",
    # Generic scam (English)
    "win-prize","you-won","youve-won","winner",
    "claim-prize","claim-reward","free-gift",
    "congratulations","survey-reward","reward-claim",
    "bitcoin-profit","crypto-profit","investment-opportunity",
    "make-money","earn-money","work-from-home",
    "get-rich","passive-income","guaranteed-profit",
    "double-your-money","triple-your-investment",
    # Gambling
    "online-casino","casino-bonus","free-spins","no-deposit",
    "betting-tips","sports-betting","live-casino",
    # Romance / dating scams
    "dating-site","find-love","meet-singles","hookup",
    "adult-dating","lonely-wives","horny-singles",
    # Tech support scams
    "virus-detected","computer-infected","security-alert",
    "call-support","remove-virus","fix-computer",
    "microsoft-support","apple-support","tech-support",
    # Survey scams
    "survey-prize","complete-survey","answer-questions",
    "opinion-reward","paid-survey",
    # Lottery / sweepstakes
    "lottery-winner","sweepstakes","raffle-winner",
    "million-dollars","inheritance","unclaimed-prize",
]


# ════════════════════════════════════════════════════════════
# COMPILE PATTERNS
# ════════════════════════════════════════════════════════════
def _compile(keywords, short_len=5):
    """Build an alternation pattern with TOKEN-BOUNDARY matching for short
    single-word keywords."""
    LETTER = r"[a-z\u00e7\u011f\u0131\u00f6\u015f\u00fc]"  # a-z + çğıöşü
    parts = []
    for kw in keywords:
        if any(c in kw for c in r'\.^$*+?{}[]|()/'):
            parts.append(kw)
        elif " " in kw or "-" in kw:
            parts.append(re.escape(kw))
        elif len(kw) < short_len:
            parts.append(r"(?<!" + LETTER + r")" + re.escape(kw) + r"(?!" + LETTER + r")")
        else:
            parts.append(re.escape(kw))
    return "|".join(parts)

PATTERN_PHISHING = _compile(PHISHING_KEYWORDS)
PATTERN_MALWARE  = _compile(MALWARE_KEYWORDS)
PATTERN_SCAM     = _compile(SCAM_KEYWORDS)


# ════════════════════════════════════════════════════════════
# RECLASSIFICATION FUNCTION
# ════════════════════════════════════════════════════════════
def reclassify_all(df, class_col="class_final", url_col="url", source_col="source"):
    """
    Reclassify ALL rows using URL keywords + source confidence.

    Logic per row:
      1. Compute keyword signal (phishing / malware / scam / none)
      2. Check source confidence:
         - High-confidence source (PhishTank, URLhaus, OpenPhish):
             * Keep source label (trust source over keywords)
         - Low-confidence / untyped source (USOM, OTX):
             * Use keyword label if available
             * Otherwise: other_malicious
      3. Result stored in 'class_final'

    Parameters
    ----------
    df         : DataFrame
    class_col  : existing class column (will be used as fallback)
    url_col    : URL column
    source_col : source column

    Returns
    -------
    df with columns: class_final, reclassify_reason
    """
    df = df.copy()

    # Handle case where input might already have class_final
    if class_col not in df.columns:
        # Try common alternatives
        if "class" in df.columns:
            class_col = "class"
        elif "label" in df.columns:
            class_col = "label"
        else:
            raise ValueError(f"No class column found. Available: {df.columns.tolist()}")

    urls = df[url_col].str.lower().fillna("")

    # Vectorised keyword hits
    phish_hit   = urls.str.contains(PATTERN_PHISHING, regex=True, na=False)
    malware_hit = urls.str.contains(PATTERN_MALWARE,  regex=True, na=False) & ~phish_hit
    scam_hit    = urls.str.contains(PATTERN_SCAM,     regex=True, na=False) & ~phish_hit & ~malware_hit
    no_hit      = ~phish_hit & ~malware_hit & ~scam_hit

    # Keyword-derived label series
    kw_label = pd.Series("other_malicious", index=df.index)
    kw_label[phish_hit]   = "phishing"
    kw_label[malware_hit] = "malware"
    kw_label[scam_hit]    = "scam"

    # Source trust mask
    src = df[source_col].fillna("__default__")
    trusted_mask = src.isin(KEEP_SOURCE_LABEL)
    reclassify_mask = src.isin(RECLASSIFY_ALL_SOURCES)
    unknown_mask = ~trusted_mask & ~reclassify_mask

    # Build class_final
    class_final  = df[class_col].copy()
    reason       = pd.Series("kept_source", index=df.index)

    # RULE 1: trusted source → always keep original label
    # (no changes for trusted_mask rows)

    # RULE 2: USOM / OTX → always use keyword (or other_malicious)
    class_final[reclassify_mask] = kw_label[reclassify_mask]
    reason[reclassify_mask]      = "reclassified_low_conf_source"

    # RULE 3: unknown source → use keyword if hit, else keep original
    kw_improves = unknown_mask & ~no_hit
    class_final[kw_improves] = kw_label[kw_improves]
    reason[kw_improves]      = "reclassified_unknown_source"

    # RULE 4: no keyword signal anywhere → always keep original
    class_final[no_hit] = df.loc[no_hit, class_col]
    reason[no_hit & ~trusted_mask] = "kept_no_signal"

    df["class_final"]       = class_final
    df["reclassify_reason"] = reason

    # ── Report ──────────────────────────────────────────────
    total = len(df)
    changed = (df[class_col] != df["class_final"]).sum()

    print(f"\n{'='*70}")
    print(f"RECLASSIFICATION COMPLETE")
    print(f"{'='*70}")
    print(f"\nTotal rows processed: {total:,}")
    print(f"Rows changed: {changed:,} ({changed/total*100:.1f}%)")
    print(f"Rows unchanged: {total-changed:,} ({(total-changed)/total*100:.1f}%)")

    print(f"\n{'-'*70}")
    print(f"REASON BREAKDOWN")
    print(f"{'-'*70}")
    for reason_val, count in df["reclassify_reason"].value_counts().items():
        print(f"  {reason_val:<35s}: {count:>10,}  ({count/total*100:>5.1f}%)")

    print(f"\n{'-'*70}")
    print(f"CLASS DISTRIBUTION (class_final)")
    print(f"{'-'*70}")
    for cls in sorted(df["class_final"].unique()):
        new_count = (df["class_final"] == cls).sum()
        old_count = (df[class_col] == cls).sum()
        delta = new_count - old_count
        sign = "+" if delta > 0 else ""
        print(f"  {cls:<20s}: {new_count:>10,}  ({new_count/total*100:>5.1f}%)  "
              f"[{sign}{delta:>8,} vs original]")

    print(f"\n{'='*70}")
    return df


# ════════════════════════════════════════════════════════════
# LABEL MATCH CHECKER
# ════════════════════════════════════════════════════════════
def check_label_match(df, original_col="class", final_col="class_final"):
    """
    Analyze agreement between original label and reclassified label.

    Provides:
    - Overall agreement rate
    - Per-class agreement rates
    - Confusion matrix
    - Detailed mismatch analysis
    - Agreement flag column

    Parameters
    ----------
    df : DataFrame with both original_col and final_col
    original_col : name of original classification column
    final_col : name of reclassified column

    Returns
    -------
    df with added 'label_match' column (True/False)
    """
    print(f"\n{'='*70}")
    print(f"LABEL vs CLASS_FINAL MATCH ANALYSIS")
    print(f"{'='*70}")

    # Add match flag
    df['label_match'] = df[original_col] == df[final_col]

    total = len(df)
    matches = df['label_match'].sum()
    mismatches = total - matches

    # Overall statistics
    print(f"\n{'─'*70}")
    print(f"OVERALL AGREEMENT")
    print(f"{'─'*70}")
    print(f"Total rows:        {total:>10,}")
    print(f"Matches:           {matches:>10,}  ({matches/total*100:>5.1f}%)")
    print(f"Mismatches:        {mismatches:>10,}  ({mismatches/total*100:>5.1f}%)")

    # Per-class agreement
    print(f"\n{'─'*70}")
    print(f"PER-CLASS AGREEMENT (Original → Final)")
    print(f"{'─'*70}")
    print(f"{'Original Class':<20s} {'Total':>10s} {'Matches':>10s} {'Match %':>10s} {'Changed':>10s}")
    print(f"{'─'*70}")

    for cls in sorted(df[original_col].unique()):
        cls_mask = df[original_col] == cls
        cls_total = cls_mask.sum()
        cls_matches = (cls_mask & df['label_match']).sum()
        cls_changed = cls_total - cls_matches
        match_pct = cls_matches / cls_total * 100 if cls_total > 0 else 0

        print(f"{cls:<20s} {cls_total:>10,} {cls_matches:>10,} {match_pct:>9.1f}% {cls_changed:>10,}")

    # Confusion/transition matrix
    print(f"\n{'─'*70}")
    print(f"TRANSITION MATRIX (Original → Final)")
    print(f"{'─'*70}")

    # Create cross-tabulation
    transition = pd.crosstab(
        df[original_col],
        df[final_col],
        rownames=['Original'],
        colnames=['Final'],
        margins=True
    )
    print(transition.to_string())

    # Detailed mismatch analysis
    if mismatches > 0:
        print(f"\n{'─'*70}")
        print(f"MISMATCH DETAILS (Top 20 transitions)")
        print(f"{'─'*70}")
        print(f"{'Original':<20s} → {'Final':<20s} {'Count':>10s} {'% of Total':>10s}")
        print(f"{'─'*70}")

        mismatch_df = df[~df['label_match']]
        transitions = mismatch_df.groupby([original_col, final_col]).size().reset_index(name='count')
        transitions = transitions.sort_values('count', ascending=False).head(20)

        for _, row in transitions.iterrows():
            orig = row[original_col]
            final = row[final_col]
            count = row['count']
            pct = count / total * 100
            print(f"{orig:<20s} → {final:<20s} {count:>10,} {pct:>9.1f}%")

    # Analysis by reclassify reason
    if 'reclassify_reason' in df.columns:
        print(f"\n{'─'*70}")
        print(f"AGREEMENT BY RECLASSIFICATION REASON")
        print(f"{'─'*70}")
        print(f"{'Reason':<35s} {'Total':>10s} {'Matches':>10s} {'Match %':>10s}")
        print(f"{'─'*70}")

        for reason in df['reclassify_reason'].unique():
            reason_mask = df['reclassify_reason'] == reason
            reason_total = reason_mask.sum()
            reason_matches = (reason_mask & df['label_match']).sum()
            match_pct = reason_matches / reason_total * 100 if reason_total > 0 else 0

            print(f"{reason:<35s} {reason_total:>10,} {reason_matches:>10,} {match_pct:>9.1f}%")

    # Source-specific analysis
    if 'source' in df.columns:
        print(f"\n{'─'*70}")
        print(f"AGREEMENT BY SOURCE (Top 10)")
        print(f"{'─'*70}")
        print(f"{'Source':<20s} {'Total':>10s} {'Matches':>10s} {'Match %':>10s}")
        print(f"{'─'*70}")

        source_stats = []
        for source in df['source'].value_counts().head(10).index:
            source_mask = df['source'] == source
            source_total = source_mask.sum()
            source_matches = (source_mask & df['label_match']).sum()
            match_pct = source_matches / source_total * 100 if source_total > 0 else 0
            source_stats.append((source, source_total, source_matches, match_pct))

        for source, total, matches, pct in source_stats:
            print(f"{source:<20s} {total:>10,} {matches:>10,} {pct:>9.1f}%")

    print(f"\n{'='*70}")

    return df


# ════════════════════════════════════════════════════════════
# EXPORT MISMATCH ANALYSIS
# ════════════════════════════════════════════════════════════
def export_mismatch_analysis(df, original_col, final_col, output_prefix="mismatch_analysis"):
    """Export detailed mismatch analysis to CSV files"""

    mismatches = df[df[original_col] != df[final_col]].copy()

    if len(mismatches) == 0:
        print(f"\n✓ No mismatches found - all labels agree!")
        return

    # 1. Full mismatch dataset
    mismatch_file = f"{output_prefix}_full.csv"
    cols = ['url', 'source', original_col, final_col, 'reclassify_reason', 'label_match']
    if 'fold' in df.columns:
        cols.append('fold')

    export_cols = [c for c in cols if c in mismatches.columns]
    mismatches[export_cols].to_csv(mismatch_file, index=False)
    print(f"\n✓ Exported {len(mismatches):,} mismatches → {mismatch_file}")

    # 2. Mismatch summary by transition
    transition_summary = mismatches.groupby([original_col, final_col]).agg({
        'url': 'count',
        'source': lambda x: x.value_counts().to_dict()
    }).rename(columns={'url': 'count'})
    transition_summary = transition_summary.sort_values('count', ascending=False)

    summary_file = f"{output_prefix}_summary.csv"
    transition_summary.to_csv(summary_file)
    print(f"✓ Exported transition summary → {summary_file}")

    # 3. Sample mismatches for each transition type (for manual review)
    samples = []
    for (orig, final), group in mismatches.groupby([original_col, final_col]):
        sample_group = group.head(10)  # Top 10 examples per transition
        samples.append(sample_group)

    if samples:
        sample_df = pd.concat(samples, ignore_index=True)
        sample_file = f"{output_prefix}_samples.csv"
        sample_df[export_cols].to_csv(sample_file, index=False)
        print(f"✓ Exported sample mismatches (10 per transition) → {sample_file}")

    return mismatches


# ════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ════════════════════════════════════════════════════════════
def main():
    """Main execution function"""

    print("="*70)
    print("TurkMAL Dataset Reclassification")
    print("="*70)

    # File paths
    input_file = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/dataset_with_folds_v10.csv"
    output_file = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/dataset_with_folds_v11.csv"

    # Check if input file exists
    if not Path(input_file).exists():
        print(f"\n❌ ERROR: Input file '{input_file}' not found!")
        print(f"   Please ensure the file is in the current directory.")
        return

    # Load dataset
    print(f"\n[1] Loading dataset: {input_file}")
    df = pd.read_csv(input_file, low_memory=False)
    print(f"    ✓ Loaded {len(df):,} rows")
    print(f"    ✓ Columns: {df.columns.tolist()[:10]}...")  # Show first 10 columns

    # Identify columns
    url_col = "url" if "url" in df.columns else df.columns[0]
    source_col = "source" if "source" in df.columns else None

    # Find class column
    class_col = None
    for col in ["class_final", "class", "label", "label_enc"]:
        if col in df.columns:
            class_col = col
            break

    if class_col is None:
        print(f"\n❌ ERROR: No class column found in dataset!")
        print(f"   Available columns: {df.columns.tolist()}")
        return

    if source_col is None:
        print(f"\n⚠️  WARNING: No 'source' column found. Using default source handling.")
        df["source"] = "__default__"
        source_col = "source"

    print(f"\n[2] Using columns:")
    print(f"    URL column:    {url_col}")
    print(f"    Class column:  {class_col}")
    print(f"    Source column: {source_col}")

    # Show original distribution
    print(f"\n[3] Original class distribution:")
    for cls, count in df[class_col].value_counts().items():
        print(f"    {cls:<20s}: {count:>10,}  ({count/len(df)*100:>5.1f}%)")

    # Apply reclassification
    print(f"\n[4] Applying keyword-based reclassification...")
    df_reclassified = reclassify_all(
        df,
        class_col=class_col,
        url_col=url_col,
        source_col=source_col
    )

    # Save results
    print(f"\n[5] Saving reclassified dataset: {output_file}")
    df_reclassified.to_csv(output_file, index=False)
    print(f"    ✓ Saved {len(df_reclassified):,} rows")

    # Summary statistics
    print(f"\n{'='*70}")
    print(f"SUMMARY")
    print(f"{'='*70}")
    print(f"Input file:  {input_file}")
    print(f"Output file: {output_file}")
    print(f"Total rows:  {len(df_reclassified):,}")

    if "class_final" in df_reclassified.columns:
        changed = (df[class_col] != df_reclassified["class_final"]).sum()
        print(f"Changed:     {changed:,} ({changed/len(df)*100:.1f}%)")
        print(f"Unchanged:   {len(df)-changed:,} ({(len(df)-changed)/len(df)*100:.1f}%)")

    print(f"\n✅ Reclassification complete!")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

TurkMAL Dataset Reclassification

[1] Loading dataset: /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/dataset_with_folds_v10.csv
    ✓ Loaded 1,239,308 rows
    ✓ Columns: ['url', 'source', 'tld', 'class_final', 'class_enc', 'label', 'label_enc', 'phishing_score', 'malware_score', 'scam_score']...

[2] Using columns:
    URL column:    url
    Class column:  class_final
    Source column: source

[3] Original class distribution:
    phishing            :    512,802  ( 41.4%)
    other_malicious     :    425,518  ( 34.3%)
    benign              :    190,740  ( 15.4%)
    malware             :     96,768  (  7.8%)
    scam                :     13,480  (  1.1%)

[4] Applying keyword-based reclassification...

RECLASSIFICATION COMPLETE

Total rows processed: 1,239,308
Rows changed: 0 (0.0%)
Rows unchanged: 1,239,308 (100.0%)

----------------------------------------------------------------------
REASON BREAKDOWN
------------------------------------------------------------

# ============================================================
# STEP 0A — BENIGN-SET PURITY CHECK (TUMC)
# ============================================================
# Answers the reviewer question: "Are your benign samples truly
# benign?" Common Crawl URLs are ASSUMED benign because they
# appeared in a broad web crawl; this script measures how many
# are actually suspect.
#
# THREE CHECKS, in increasing strength:
#   C1  Internal overlap — is any benign URL also in the
#       malicious set? (fully offline, MUST be zero)
#   C2  Domain overlap — does any benign URL share a registered
#       domain with a malicious URL? (offline; flags ambiguity)
#   C3  Blocklist cross-check — do benign URLs/domains appear on
#       public blocklists? (OPTIONAL, needs network + the lists)
#
# HONEST SCOPE: C1+C2 are offline and definitive for overlap.
# C3 requires you to supply blocklist files (URLhaus, OpenPhish,
# a hosts blocklist) — it does NOT call any live API here, to
# keep it reproducible and avoid rate limits / privacy issues.
#
# Output: step0a_benign_purity_report.txt + flagged CSVs
# ============================================================

In [ ]:
%cd "/content/mydrive/MyDrive/Colab Notebooks/Research/New_Dataset_May"

/content/mydrive/MyDrive/Colab Notebooks/Research/New_Dataset_May


In [ ]:
!pip install tldextract -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.7 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import tldextract

try:
    _extract = tldextract.TLDExtract()
    _ = _extract("test.com.tr")
except Exception:
    _extract = tldextract.TLDExtract(suffix_list_urls=())

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
# Final merged dataset with the 'label' (benign/malicious) and 'class_final'
for cand in ["dataset_with_folds_v11.csv", "features_full_v12.csv", "dataset_with_folds_v10.csv",
             "dataset_with_folds_v9.csv"]:
    DATASET = os.path.join(DATA_DIR, cand)
    if os.path.exists(DATASET): break

# Leave empty to skip C3. These should be files you DOWNLOAD yourself from
# the official sources and place in DATA_DIR/blocklists/.
BLOCKLIST_FILES = [
    # os.path.join(DATA_DIR, "blocklists", "urlhaus.txt"),
    # os.path.join(DATA_DIR, "blocklists", "openphish.txt"),
    # os.path.join(DATA_DIR, "blocklists", "hosts_blocklist.txt"),
]

OUT = os.path.join(DATA_DIR, "step0a_benign_purity_report.txt")
report = []
def log(s):
    print(s); report.append(s)

def reg_domain(u):
    u = str(u)
    if not u.startswith("http"): u = "http://" + u
    e = _extract(u)
    return f"{e.domain}.{e.suffix}".lower() if e.suffix else e.domain.lower()

log("="*70)
log("STEP 0A — BENIGN-SET PURITY CHECK")
log("="*70)

df = pd.read_csv(DATASET, low_memory=False)
df["url_norm"] = df["url"].astype(str).str.strip().str.lower()
benign = df[df["label"] == "benign"].copy()
malic  = df[df["label"] == "malicious"].copy()
log(f"  Benign URLs:    {len(benign):,}")
log(f"  Malicious URLs: {len(malic):,}")

# ── C1: exact-URL overlap (must be zero) ─────────────────────
log("\n[C1] Exact-URL overlap (benign ∩ malicious)")
mal_urls = set(malic["url_norm"])
ben_in_mal = benign[benign["url_norm"].isin(mal_urls)]
log(f"  Benign URLs that are ALSO labelled malicious: {len(ben_in_mal):,} "
    f"({len(ben_in_mal)/len(benign)*100:.4f}%)")
if len(ben_in_mal) > 0:
    log("  ⚠ NON-ZERO OVERLAP — these URLs have contradictory labels.")
    log("    Action: remove from benign or resolve label. Sample:")
    for u in ben_in_mal["url"].head(10): log(f"      {u}")
    ben_in_mal[["url","class_final"]].to_csv(
        os.path.join(DATA_DIR,"flagged_url_overlap.csv"), index=False)
else:
    log("  ✓ Zero exact-URL overlap — benign and malicious sets are disjoint.")

# ── C2: registered-domain overlap ────────────────────────────
log("\n[C2] Registered-domain overlap")
log("  (A benign URL sharing a domain with a malicious URL is")
log("   ambiguous: the domain hosts both benign and malicious paths,")
log("   OR the benign label is wrong. Worth manual review.)")
benign["reg_dom"] = benign["url"].apply(reg_domain)
malic["reg_dom"]  = malic["url"].apply(reg_domain)
mal_doms = set(malic["reg_dom"])
ben_shared = benign[benign["reg_dom"].isin(mal_doms)]
n_shared_dom = ben_shared["reg_dom"].nunique()
log(f"  Benign URLs on a domain also seen in malicious set: "
    f"{len(ben_shared):,} ({len(ben_shared)/len(benign)*100:.2f}%)")
log(f"  Distinct such domains: {n_shared_dom:,}")
if len(ben_shared) > 0:
    top = ben_shared["reg_dom"].value_counts().head(15)
    log("  Top shared domains (benign URL count):")
    for d,c in top.items(): log(f"      {d:<35s} {c}")
    ben_shared[["url","reg_dom","class_final"]].to_csv(
        os.path.join(DATA_DIR,"flagged_domain_overlap.csv"), index=False)
    log(f"  → Saved {len(ben_shared)} rows to flagged_domain_overlap.csv")
    log("    NOTE: shared domains may be legitimate (e.g. blogspot.com,")
    log("    github.io host both benign and malicious subpaths). Review")
    log("    the list; high-reputation shared hosts are expected and OK,")
    log("    but a benign URL sharing a domain with a bank-phishing URL")
    log("    is a red flag worth reclassifying.")

# ── C3: blocklist cross-check (optional, offline against files) ─
log("\n[C3] Public-blocklist cross-check")
if not BLOCKLIST_FILES:
    log("  SKIPPED — no blocklist files supplied.")
    log("  To run: download URLhaus/OpenPhish/a hosts blocklist, place in")
    log("  DATA_DIR/blocklists/, and add paths to BLOCKLIST_FILES at top.")
    log("  This converts 'assumed benign' into 'verified not-on-blocklist',")
    log("  which is the strongest answer to the reviewer question.")
else:
    block_urls, block_doms = set(), set()
    for bf in BLOCKLIST_FILES:
        if not os.path.exists(bf):
            log(f"  ⚠ missing: {bf}"); continue
        with open(bf, encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                t = line.strip().lower()
                if not t or t.startswith("#"): continue
                # hosts-file format: "0.0.0.0 domain" → take last token
                tok = t.split()[-1]
                block_urls.add(tok)
                block_doms.add(reg_domain(tok))
        log(f"  loaded {os.path.basename(bf)}")
    log(f"  blocklist URLs: {len(block_urls):,}  domains: {len(block_doms):,}")
    hit_url = benign[benign["url_norm"].isin(block_urls)]
    hit_dom = benign[benign["reg_dom"].isin(block_doms)]
    log(f"  Benign URLs on a blocklist (exact):  {len(hit_url):,} "
        f"({len(hit_url)/len(benign)*100:.4f}%)")
    log(f"  Benign URLs on a blocklist (domain): {len(hit_dom):,} "
        f"({len(hit_dom)/len(benign)*100:.4f}%)")
    contam = pd.concat([hit_url, hit_dom]).drop_duplicates("url")
    if len(contam) > 0:
        contam[["url","reg_dom","class_final"]].to_csv(
            os.path.join(DATA_DIR,"flagged_blocklist_hits.csv"), index=False)
        log(f"  → {len(contam)} benign URLs flagged as blocklist hits.")
        log("    Recommended: remove these from benign or reassign.")
    log(f"\n  REPORTABLE: \"{len(contam)/len(benign)*100:.3f}% of benign")
    log(f"  candidates appeared on a public blocklist and were removed,")
    log(f"  yielding a verified-clean benign rate of "
        f"{(1-len(contam)/len(benign))*100:.3f}%.\"")

# ── Summary ──────────────────────────────────────────────────
log("\n" + "="*70)
log("SUMMARY — what you can honestly claim")
log("="*70)
log(f"  • Benign/malicious exact-URL overlap: {len(ben_in_mal)} "
    f"({'CLEAN' if len(ben_in_mal)==0 else 'NEEDS FIX'})")
log(f"  • Benign URLs sharing a malicious domain: {len(ben_shared)} "
    f"({len(ben_shared)/len(benign)*100:.2f}%) — review flagged CSV")
log(f"  • Blocklist check: {'RUN IT with supplied lists' if not BLOCKLIST_FILES else 'done'}")
log("\n  The honest paper statement depends on these numbers. If C1=0")
log("  and C2 is small/explainable and C3 contamination is low, you can")
log("  state the benign set is verified disjoint and blocklist-clean.")

with open(OUT, "w") as fh: fh.write("\n".join(report))
print(f"\n[saved] {OUT}")

STEP 0A — BENIGN-SET PURITY CHECK
  Benign URLs:    190,740
  Malicious URLs: 1,048,568

[C1] Exact-URL overlap (benign ∩ malicious)
  Benign URLs that are ALSO labelled malicious: 0 (0.0000%)
  ✓ Zero exact-URL overlap — benign and malicious sets are disjoint.

[C2] Registered-domain overlap
  (A benign URL sharing a domain with a malicious URL is
   ambiguous: the domain hosts both benign and malicious paths,
   OR the benign label is wrong. Worth manual review.)
  Benign URLs on a domain also seen in malicious set: 10,023 (5.25%)
  Distinct such domains: 612
  Top shared domains (benign URL count):
      microsoft.com                       978
      akadns.net                          924
      amazonaws.com                       690
      apple.com                           526
      azure.com                           526
      cloudfront.net                      503
      googleapis.com                      369
      windows.net                         361
      google.com       

# ============================================================
# STEP 0A-2 — FETCH BLOCKLISTS + BENIGN CROSS-CHECK (TUMC)
# ============================================================
# Implements the C3 blocklist cross-check end-to-end: downloads
# reputable public blocklists, then checks how many of your
# "benign" (Common Crawl) URLs appear on them.
#
# This turns "assumed benign" into "verified not on any major
# blocklist", the strongest answer to the reviewer question
# "are our benign samples truly benign?"
#
# SOURCES (public, GitHub-mirrored, domain-level blocklists):
#   - blocklistproject phishing + malware
#   - mitchellkrogza Phishing.Database (ACTIVE domains)
# These are DOMAIN blocklists, so the check is domain-level.
#
# HONEST SCOPE:
#   - A hit means the benign URL's registered domain is on a
#     malicious blocklist → strong signal it is NOT benign.
#   - A non-hit does NOT prove benignity (blocklists are
#     incomplete), but "0.X% of benign URLs hit a major
#     blocklist" is a legitimate, reportable purity measure.
#   - Blocklists themselves have false positives; review hits
#     before deleting, do not auto-purge blindly.
#
# Output: blocklists/*.txt (cached), step0a2_blocklist_report.txt,
#         flagged_blocklist_hits.csv
# ============================================================

In [ ]:
import os, urllib.request, time
import pandas as pd
import tldextract

# Use tldextract with its bundled snapshot if the Public Suffix List
# cannot be fetched (keeps the run offline-safe and deterministic).
# The snapshot correctly handles Turkish multi-part ccTLDs (.com.tr,
# .gov.tr, .edu.tr), which is essential for this dataset.
try:
    _extract = tldextract.TLDExtract()
    _ = _extract("test.com.tr")          # trigger; falls back to snapshot if offline
except Exception:
    _extract = tldextract.TLDExtract(suffix_list_urls=())

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
BL_DIR = os.path.join(DATA_DIR, "blocklists")
os.makedirs(BL_DIR, exist_ok=True)
for cand in ["dataset_with_folds_v11.csv", "features_full_v12.csv", "dataset_with_folds_v11.csv"]:
    DATASET = os.path.join(DATA_DIR, cand)
    if os.path.exists(DATASET): break
OUT = os.path.join(DATA_DIR, "step0a2_blocklist_report.txt")

# (filename, url, format)  format: 'hosts' = "0.0.0.0 domain", 'plain' = "domain"
SOURCES = [
    ("blp_phishing.txt",
     "https://raw.githubusercontent.com/blocklistproject/Lists/master/phishing.txt", "hosts"),
    ("blp_malware.txt",
     "https://raw.githubusercontent.com/blocklistproject/Lists/master/malware.txt", "hosts"),
    ("krogza_phishing_active.txt",
     "https://raw.githubusercontent.com/mitchellkrogza/Phishing.Database/master/phishing-domains-ACTIVE.txt", "plain"),
]
REFRESH = False   # set True to re-download even if cached

report = []
def log(s):
    print(s); report.append(s)

def reg_domain(u):
    u = str(u)
    if not u.startswith("http"): u = "http://" + u
    e = _extract(u)
    return f"{e.domain}.{e.suffix}".lower() if e.suffix else e.domain.lower()

log("="*70)
log("STEP 0A-2 — FETCH BLOCKLISTS + BENIGN CROSS-CHECK")
log("="*70)

# ── 1. Download (cached) ─────────────────────────────────────
log("\n[1] Fetching blocklists (cached in blocklists/)")
block_domains = set()
per_source = {}
for fname, url, fmt in SOURCES:
    path = os.path.join(BL_DIR, fname)
    if REFRESH or not os.path.exists(path):
        try:
            log(f"  downloading {fname} ...")
            req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                open(path, "wb").write(r.read())
            time.sleep(1)
        except Exception as e:
            log(f"  ⚠ failed {fname}: {e}"); continue
    else:
        log(f"  cached {fname}")
    # parse
    doms = set()
    with open(path, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            t = line.strip().lower()
            if not t or t.startswith("#"): continue
            tok = t.split()[-1] if fmt == "hosts" else t
            tok = tok.lstrip("*.").strip()
            if "." in tok and " " not in tok:
                doms.add(tok)
    per_source[fname] = len(doms)
    block_domains |= doms
    log(f"      {len(doms):,} domains")
log(f"  TOTAL unique blocklist domains: {len(block_domains):,}")

if not block_domains:
    log("  No blocklists loaded — aborting."); open(OUT,"w").write("\n".join(report)); raise SystemExit

# ── 2. Cross-check benign set ────────────────────────────────
log("\n[2] Cross-checking benign URLs")
df = pd.read_csv(DATASET, low_memory=False)
benign = df[df["label"] == "benign"].copy()
log(f"  benign URLs: {len(benign):,}")
benign["reg_dom"] = benign["url"].apply(reg_domain)

hits = benign[benign["reg_dom"].isin(block_domains)]
n_hit = len(hits)
n_hit_dom = hits["reg_dom"].nunique()
rate = n_hit/len(benign)*100
log(f"  benign URLs whose domain is on a blocklist: {n_hit:,} ({rate:.4f}%)")
log(f"  distinct flagged domains: {n_hit_dom:,}")

if n_hit > 0:
    cls = "class_final" if "class_final" in hits.columns else "class"
    cols = ["url","reg_dom"] + ([cls] if cls in hits.columns else [])
    hits[cols].to_csv(os.path.join(DATA_DIR,"flagged_blocklist_hits.csv"), index=False)
    log(f"  → saved {n_hit} flagged rows to flagged_blocklist_hits.csv")
    log("  top flagged domains:")
    for d,c in hits["reg_dom"].value_counts().head(15).items():
        log(f"      {d:<40s} {c}")

# ── 3. Reportable statement ──────────────────────────────────
log("\n" + "="*70)
log("REPORTABLE STATEMENT (for the manuscript)")
log("="*70)
clean = 100 - rate
log(f"  \"The benign corpus was cross-checked against "
    f"{len(block_domains):,} domains aggregated from public phishing")
log(f"  and malware blocklists. {rate:.3f}% of benign URLs matched a")
log(f"  blocklisted domain and were flagged for removal, yielding a")
log(f"  blocklist-clean benign rate of {clean:.3f}%.\"")
log("")
log("  CAVEATS to state honestly:")
log("   - Domain-level match; blocklists are incomplete (non-hit ≠ clean).")
log("   - Blocklists carry false positives; flagged rows were reviewed,")
log("     not auto-deleted.")
log("   - Check reflects blocklist state at access time, not collection time.")

open(OUT,"w").write("\n".join(report))
print(f"\n[saved] {OUT}")

STEP 0A-2 — FETCH BLOCKLISTS + BENIGN CROSS-CHECK

[1] Fetching blocklists (cached in blocklists/)
  cached blp_phishing.txt
      190,218 domains
  cached blp_malware.txt
      435,220 domains
  cached krogza_phishing_active.txt
      391,515 domains
  TOTAL unique blocklist domains: 1,005,582

[2] Cross-checking benign URLs
  benign URLs: 190,740
  benign URLs whose domain is on a blocklist: 1,078 (0.5652%)
  distinct flagged domains: 259
  → saved 1078 flagged rows to flagged_blocklist_hits.csv
  top flagged domains:
      adnangul.av.tr                           56
      inmobi.com                               54
      smartadserver.com                        50
      adsafeprotected.com                      35
      adsrvr.org                               34
      yieldmo.com                              27
      3lift.com                                26
      doubleverify.com                         26
      omnitagjs.com                            25
      aniview.com       

# ============================================================
# STEP 0A-3 — TRIAGE FLAGGED BENIGN URLS (TUMC)
# ============================================================
# The blocklist cross-check (Step 0A-2) flagged 1,078 benign
# URLs across 259 domains. Most are ad-tech / CDN false
# positives, but at least one (adnangul.av.tr) is confirmed
# real malware. This script buckets the flagged domains so a
# human can review hundreds in minutes, then helps you produce
# a CLEANED benign set with the true contamination removed.
#
# BUCKETS (heuristic, for REVIEW not auto-decision):
#   ADTECH   — ad serving / tracking (likely false positive)
#   CLOUD    — cloud / CDN / hosting (likely shared-infra FP)
#   TR       — .tr domains (HIGH PRIORITY manual review)
#   OTHER    — everything else (manual review)
#
# The script NEVER auto-deletes. It writes a review sheet with
# a suggested action you confirm/override, then (in apply mode)
# removes only the domains you marked 'remove'.
#
# Output:
#   step0a3_triage_review.csv   (review sheet, you edit 'action')
#   features_full_v12_cleaned.csv  (apply mode, after your review)
# ============================================================

In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
FLAGGED = os.path.join(DATA_DIR, "flagged_blocklist_hits.csv")
REVIEW  = os.path.join(DATA_DIR, "step0a3_triage_review.csv")
for cand in ["dataset_with_folds_v11.csv","features_full_v12.csv", "dataset_with_folds_v10.csv"]:
    DATASET = os.path.join(DATA_DIR, cand)
    if os.path.exists(DATASET): break

MODE = "triage"          # "triage" → build review sheet; "apply" → remove marked
CLEAN_OUT = os.path.join(DATA_DIR, "features_full_v12_cleaned.csv")

# Heuristic keyword sets for bucketing (NOT for final decision)
ADTECH = ["adsrvr","adserver","adsafe","doubleverify","3lift","yieldmo",
          "smartadserver","inmobi","omnitagjs","aniview","stackadapt","smaato",
          "sharethrough","creativecdn","contextweb","adnxs","pubmatic","rubicon",
          "criteo","taboola","outbrain","moatads","scorecardresearch","casalemedia",
          "advertising","adsystem","adservice","adtech","tracking","analytics"]
CLOUD  = ["amazonaws","cloudfront","azure","windows.net","googleapis","google.com",
          "microsoft","apple","fastly","akadns","googleusercontent","office.com",
          "qq.com","epicgames","gstatic","akamai","cloudflare","herokuapp",
          "github.io","blogspot","wordpress","firebaseapp","web.app"]

def bucket(domain):
    d = domain.lower()
    if d.endswith(".tr"):                       return "TR"
    if any(k in d for k in ADTECH):             return "ADTECH"
    if any(k in d for k in CLOUD):              return "CLOUD"
    return "OTHER"

if MODE == "triage":
    print("="*70); print("STEP 0A-3 — TRIAGE FLAGGED BENIGN DOMAINS"); print("="*70)
    fl = pd.read_csv(FLAGGED)
    by_dom = (fl.groupby("reg_dom").size()
                .reset_index(name="n_urls").sort_values("n_urls", ascending=False))
    by_dom["bucket"] = by_dom["reg_dom"].apply(bucket)
    # suggested default action: review TR/OTHER, keep ADTECH/CLOUD
    by_dom["suggested"] = by_dom["bucket"].map(
        {"TR":"REMOVE?","OTHER":"REVIEW","ADTECH":"keep","CLOUD":"keep"})
    by_dom["action"] = ""    # YOU fill: 'remove' or 'keep'
    by_dom = by_dom[["reg_dom","n_urls","bucket","suggested","action"]]
    by_dom.to_csv(REVIEW, index=False)

    print(f"\n  Flagged: {len(fl)} URLs across {by_dom['reg_dom'].nunique()} domains")
    print("\n  Bucket summary (domains / URLs):")
    for b in ["TR","OTHER","ADTECH","CLOUD"]:
        sub = by_dom[by_dom["bucket"]==b]
        print(f"    {b:<7s}: {len(sub):>4d} domains, {sub['n_urls'].sum():>5d} URLs")
    print(f"\n  Review sheet → {REVIEW}")
    print("  ACTION REQUIRED:")
    print("   1. Open the sheet. Focus on TR and OTHER buckets first")
    print("      (these contain real contamination like adnangul.av.tr).")
    print("   2. For each domain, set action='remove' (true malicious) or")
    print("      'keep' (false positive). ADTECH/CLOUD default to keep but")
    print("      override any you find suspicious.")
    print("   3. Inspect URL strings only; do not open in a live browser.")
    print("   4. Set MODE='apply' and rerun to produce the cleaned dataset.")
    print("\n  PRE-FILLED for you based on confirmation:")
    print("    adnangul.av.tr is CONFIRMED malware → mark 'remove'.")

    # show TR + OTHER for immediate review
    print("\n  TR-bucket domains (review ALL of these):")
    for _,r in by_dom[by_dom["bucket"]=="TR"].iterrows():
        print(f"    {r['reg_dom']:<40s} {r['n_urls']} urls")
    print("\n  OTHER-bucket top 25 (review):")
    for _,r in by_dom[by_dom["bucket"]=="OTHER"].head(25).iterrows():
        print(f"    {r['reg_dom']:<40s} {r['n_urls']} urls")

elif MODE == "apply":
    print("="*70); print("STEP 0A-3 — APPLY CLEANING"); print("="*70)
    rev = pd.read_csv(REVIEW)
    rev["action"] = rev["action"].astype(str).str.strip().str.lower()
    remove_doms = set(rev[rev["action"]=="remove"]["reg_dom"])
    if not remove_doms:
        print("  No domains marked 'remove'. Edit the review sheet first.")
        raise SystemExit
    print(f"  Domains marked for removal: {len(remove_doms)}")
    for d in sorted(remove_doms): print(f"    - {d}")

    df = pd.read_csv(DATASET, low_memory=False)
    import tldextract
    try:
        _ex = tldextract.TLDExtract(); _ex("test.com.tr")
    except Exception:
        _ex = tldextract.TLDExtract(suffix_list_urls=())
    def rd(u):
        u=str(u);
        if not u.startswith("http"): u="http://"+u
        e=_ex(u); return f"{e.domain}.{e.suffix}".lower() if e.suffix else e.domain.lower()

    before = len(df)
    # Only remove BENIGN rows on those domains (don't touch malicious)
    df["_rd"] = df["url"].apply(rd)
    mask = (df["label"]=="benign") & (df["_rd"].isin(remove_doms))
    n_removed = mask.sum()
    df = df[~mask].drop(columns=["_rd"])
    df.to_csv(CLEAN_OUT, index=False)
    print(f"\n  Removed {n_removed} benign URLs ({n_removed/before*100:.4f}% of corpus)")
    print(f"  Cleaned dataset: {before:,} → {len(df):,} rows")
    print(f"  → {CLEAN_OUT}")
    print("\n  REMINDER: re-run feature selection (2E) and Step 3 on the")
    print("  cleaned file if the removal is material. For a removal this")
    print("  small the impact on metrics will be negligible, but the")
    print("  cleaned provenance is what you report.")

else:
    print("Set MODE to 'triage' or 'apply'.")

STEP 0A-3 — TRIAGE FLAGGED BENIGN DOMAINS

  Flagged: 1078 URLs across 259 domains

  Bucket summary (domains / URLs):
    TR     :   86 domains,   141 URLs
    OTHER  :  155 domains,   538 URLs
    ADTECH :   18 domains,   399 URLs
    CLOUD  :    0 domains,     0 URLs

  Review sheet → /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/step0a3_triage_review.csv
  ACTION REQUIRED:
   1. Open the sheet. Focus on TR and OTHER buckets first
      (these contain real contamination like adnangul.av.tr).
   2. For each domain, set action='remove' (true malicious) or
      'keep' (false positive). ADTECH/CLOUD default to keep but
      override any you find suspicious.
   3. Inspect URL strings only; do not open in a live browser.
   4. Set MODE='apply' and rerun to produce the cleaned dataset.

  PRE-FILLED for you based on confirmation:
    adnangul.av.tr is CONFIRMED malware → mark 'remove'.

  TR-bucket domains (review ALL of these):
    adnangul.av.tr                       

# ============================================================
# STEP 0B — LEXICON VALIDATION (TUMC)
# ============================================================
# Moves the Turkish threat lexicons from "curated, unvalidated"
# to "empirically validated" by measuring, on the real data:
#
#   COVERAGE  — % of URLs in each class that contain >=1 term
#               from the matching lexicon (recall-like).
#   PRECISION — of URLs that hit a lexicon, what % are actually
#               in the target class vs benign (false-positive view).
#   BENIGN FP — % of BENIGN URLs that hit each lexicon (should be
#               LOW; this is what the cleaning/token-matching buys).
#   DISCRIMINATION — coverage on target class minus coverage on
#               benign (higher = more discriminative).
#
# Uses TOKEN-AWARE matching (same as the feature code) so results
# reflect how the features actually behave.
#
# HONEST SCOPE: this validates the lexicons against THIS dataset's
# labels. It is internal validation. It does NOT prove external
# completeness (impossible) and shares the keyword-circularity
# caveat for classes whose labels were partly assigned via
# overlapping lexicons — reported explicitly below.
#
# Output: step0b_lexicon_validation.csv + report
# ============================================================


In [ ]:
import os, re, warnings
from urllib.parse import unquote
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11.csv","features_full_v12.csv","dataset_with_folds_v10.csv"]:
    DATASET = os.path.join(DATA_DIR, cand)
    if os.path.exists(DATASET): break
OUT_CSV = os.path.join(DATA_DIR, "step0c_lexicon_validation.csv")
OUT_TXT = os.path.join(DATA_DIR, "step0c_lexicon_validation_report.txt")

import sys; sys.path.insert(0, DATA_DIR); sys.path.insert(0, ".")
from turkish_lexicons import (TR_PHISHING, TR_SCAM, TR_MALWARE,
                              TR_DEFACEMENT, TR_URGENCY)
from lexicon_cleaner import clean_lexicon
# clean at load (idempotent if file already clean)
LEXICONS = {
    "phishing":   clean_lexicon(TR_PHISHING),
    "scam":       clean_lexicon(TR_SCAM),
    "malware":    clean_lexicon(TR_MALWARE),
    "urgency":    clean_lexicon(TR_URGENCY),   # cross-category, no single class
}
# Map each lexicon to the class it should cover (urgency spans all malicious)
LEX_TO_CLASS = {"phishing":"phishing","scam":"scam","malware":"malware"}

report=[]
def log(s): print(s); report.append(s)

def tokenize(u):
    return [t for t in re.split(r"[^a-zçğıöşü0-9]+", unquote(str(u).lower())) if len(t)>=2]

def hits_lexicon(url, lexicon):
    """Token-aware: True if URL contains >=1 lexicon term (same logic as features)."""
    u = unquote(str(url).lower())
    toks = tokenize(u); ts = set(toks); j = u.replace("/","").replace(".","")
    for term in lexicon:
        if " " in term or "-" in term:
            if term.replace("-","") in j or term in j: return True
        else:
            if term in ts or (len(term)>=5 and any(t.startswith(term) for t in toks)):
                return True
    return False

log("="*70)
log("STEP 0C — LEXICON VALIDATION")
log(f"  Source: {os.path.basename(DATASET)}")
log("="*70)

df = pd.read_csv(DATASET, low_memory=False)
col = "class_final" if "class_final" in df.columns else "class"
log(f"  Rows: {len(df):,}")
for name, lex in LEXICONS.items():
    log(f"  {name} lexicon: {len(lex)} terms")

# Precompute per-lexicon hit flags (sample if huge, for speed)
SAMPLE = min(len(df), 300_000)   # validate on a large sample for tractability
if len(df) > SAMPLE:
    dfs = df.sample(n=SAMPLE, random_state=42)
    log(f"  (validating on {SAMPLE:,}-row stratified-ish sample for speed)")
else:
    dfs = df

log("\n  Computing lexicon hits (token-aware)...")
hit_flags = {}
for name, lex in LEXICONS.items():
    hit_flags[name] = dfs["url"].apply(lambda u: hits_lexicon(u, lex)).values
    log(f"    {name} done")

classes = [c for c in ["benign","phishing","malware","scam","other_malicious"]
           if c in dfs[col].unique()]
cls_arr = dfs[col].values

# ── Coverage + benign FP + discrimination per lexicon ────────
rows=[]
log("\n" + "="*70)
log("VALIDATION RESULTS")
log("="*70)
for name, flags in hit_flags.items():
    benign_mask = cls_arr == "benign"
    benign_fp = flags[benign_mask].mean()*100 if benign_mask.sum() else float("nan")
    target = LEX_TO_CLASS.get(name)
    if target and target in classes:
        tgt_mask = cls_arr == target
        coverage = flags[tgt_mask].mean()*100
        discrimination = coverage - benign_fp
        # precision: of all hits, fraction that are target class
        hit_idx = flags
        n_hit = hit_idx.sum()
        prec = (flags & tgt_mask).sum()/n_hit*100 if n_hit else float("nan")
        log(f"\n  [{name}] → target class '{target}'")
        log(f"    Coverage (recall): {coverage:.1f}% of {target} URLs contain a term")
        log(f"    Benign FP rate:    {benign_fp:.2f}% of benign URLs hit (lower=better)")
        log(f"    Discrimination:    {discrimination:+.1f} pp (coverage - benign FP)")
        rows.append({"lexicon":name,"target":target,"coverage_pct":round(coverage,2),
                     "benign_fp_pct":round(benign_fp,3),
                     "discrimination_pp":round(discrimination,2)})
    else:
        # urgency: report coverage across ALL malicious vs benign
        mal_mask = cls_arr != "benign"
        cov_mal = flags[mal_mask].mean()*100
        log(f"\n  [{name}] → cross-category (all malicious)")
        log(f"    Coverage on malicious: {cov_mal:.1f}%")
        log(f"    Benign FP rate:        {benign_fp:.2f}%")
        log(f"    Discrimination:        {cov_mal-benign_fp:+.1f} pp")
        rows.append({"lexicon":name,"target":"all_malicious","coverage_pct":round(cov_mal,2),
                     "benign_fp_pct":round(benign_fp,3),
                     "discrimination_pp":round(cov_mal-benign_fp,2)})

val_df = pd.DataFrame(rows)
val_df.to_csv(OUT_CSV, index=False)

# ── Cross-lexicon coverage table (what % of each class is covered
#    by ANY lexicon) — overall reach ───────────────────────────
log("\n" + "="*70)
log("OVERALL REACH — % of each class hitting ANY threat lexicon")
log("="*70)
any_threat = np.zeros(len(dfs), dtype=bool)
for name in ["phishing","scam","malware"]:
    any_threat |= hit_flags[name]
for c in classes:
    m = cls_arr == c
    log(f"  {c:<18s}: {any_threat[m].mean()*100:5.1f}% contain >=1 threat term")

# ── Honest interpretation ────────────────────────────────────
log("\n" + "="*70)
log("INTERPRETATION (for our manuscript)")
log("="*70)
log("  Report coverage + benign-FP as the empirical validation of")
log("  the lexicons. A high discrimination (coverage >> benign FP)")
log("  shows the lexicon is precise, not noise.")
log("")
log("  STILL STATE in limitations:")
log("   - This is INTERNAL validation against TUMC labels.")
log("   - Phishing/scam/malware labels were partly assigned via")
log("     overlapping lexicons → coverage on those classes is")
log("     partly circular. The benign-FP rate is NOT circular and")
log("     is the cleanest validity signal (it shows precision on")
log("     data the lexicon did not help label).")
log("   - Coverage < 100% is expected and honest; lexicons are")
log("     inherently incomplete.")
log("   - Consider cross-referencing terms against USOM advisories")
log("     and sector security bulletins for external grounding.")

open(OUT_TXT,"w").write("\n".join(report))
print(f"\n[saved] {OUT_CSV}")
print(f"[saved] {OUT_TXT}")

STEP 0C — LEXICON VALIDATION
  Source: dataset_with_folds_v11.csv
  Rows: 1,239,308
  phishing lexicon: 154 terms
  scam lexicon: 105 terms
  malware lexicon: 55 terms
  urgency lexicon: 47 terms
  (validating on 300,000-row stratified-ish sample for speed)

  Computing lexicon hits (token-aware)...
    phishing done
    scam done
    malware done
    urgency done

VALIDATION RESULTS

  [phishing] → target class 'phishing'
    Coverage (recall): 8.3% of phishing URLs contain a term
    Benign FP rate:    1.38% of benign URLs hit (lower=better)
    Discrimination:    +6.9 pp (coverage - benign FP)

  [scam] → target class 'scam'
    Coverage (recall): 34.2% of scam URLs contain a term
    Benign FP rate:    0.57% of benign URLs hit (lower=better)
    Discrimination:    +33.6 pp (coverage - benign FP)

  [malware] → target class 'malware'
    Coverage (recall): 1.5% of malware URLs contain a term
    Benign FP rate:    0.27% of benign URLs hit (lower=better)
    Discrimination:    +1.2 p

In [ ]:
!pip install tldextract -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd  "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"

/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May


# ============================================================
# STEP 0C — SOURCE AUDIT, EMPTY-SOURCE CLEANING, AND
#           BENIGN SOURCE-COMPOSITION ANALYSIS (TUMC)
# ============================================================
# Three jobs, run in safe order:
#
#  PART 1 — AUDIT (no deletion): how many rows have an empty/
#    missing source, broken down by class and by binary label,
#    plus the full inventory of source values. You REVIEW this.
#
#  PART 2 — CLEAN: remove rows with empty source, write a NEW
#    file (original untouched), report before/after counts and
#    the per-class impact so the change is documented.
#
#  PART 3 — BENIGN COMPOSITION: of the benign rows, how many come
#    from Common Crawl vs Umbrella vs other sources, and what
#    fraction of each is .tr / Turkish. This quantifies the
#    source-provenance bias for honest reporting.
#
# SET MODE: "audit" first (look, don't touch). After reviewing,
# set "clean" to produce the cleaned file.
# ============================================================

In [ ]:
import os, re
import pandas as pd
import tldextract

MODE = "apply"          # "audit" -> inspect only; "relabel" -> tag empties as 'unknown'
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11.csv","features_full_v13_srcnorm.csv","features_full_v13.csv","features_full_v12_inductive.csv","features_full_v12.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break
CLEAN_OUT = os.path.join(DATA_DIR, os.path.basename(INPUT).replace(".csv","_srclabeled.csv"))
REPORT = os.path.join(DATA_DIR, "step0d_source_report.txt")

# Source-name patterns (lowercased substring match) → canonical bucket
SOURCE_BUCKETS = {
    "common_crawl": ["common", "commoncrawl", "cc", "crawl", "cdx", "commoncrawl_tr"],
    "umbrella":     ["umbrella", "cisco"],
    "tranco":       ["tranco"],
    "majestic":     ["majestic"],
    "phishtank":    ["phishtank"],
    "openphish":    ["openphish"],
    "urlhaus":      ["urlhaus"],
    "usom":         ["usom", "USOM"],
    "phishstats":   ["phishstats"],
}

_ex = tldextract.TLDExtract()
try: _ex("test.com.tr")
except Exception: _ex = tldextract.TLDExtract(suffix_list_urls=())

def is_tr(url):
    u = str(url)
    e = _ex(u if u.startswith("http") else "http://"+u)
    return e.suffix.endswith("tr")

def bucket(src):
    s = str(src).strip().lower()
    if s in ("", "nan", "none", "null"): return "(empty)"
    for canon, pats in SOURCE_BUCKETS.items():
        if any(p in s for p in pats): return canon
    return f"other:{s[:25]}"

report=[]
def log(s): print(s); report.append(s)

log("="*70); log(f"STEP 0D — SOURCE AUDIT  (mode={MODE})"); log(f"  file: {os.path.basename(INPUT)}"); log("="*70)

df = pd.read_csv(INPUT, low_memory=False)
log(f"  total rows: {len(df):,}")
if "source" not in df.columns:
    log("  ⚠ no 'source' column — cannot proceed."); open(REPORT,"w").write("\n".join(report)); raise SystemExit

src = df["source"].astype(str).str.strip().str.lower()
empty_mask = src.isin(["", "nan", "none", "null"]) | df["source"].isna()
n_empty = int(empty_mask.sum())
label_col = "label" if "label" in df.columns else None
class_col = "class_final" if "class_final" in df.columns else ("class" if "class" in df.columns else None)

# ── PART 1: AUDIT ────────────────────────────────────────────
log("\n[PART 1] EMPTY-SOURCE AUDIT")
log(f"  rows with empty/missing source: {n_empty:,} ({n_empty/len(df)*100:.2f}%)")
if class_col:
    log("\n  Empty-source rows by class:")
    ec = df[empty_mask][class_col].value_counts()
    for c,n in ec.items(): log(f"    {str(c):<18s}: {n:,}")
    log("\n  For comparison — FULL dataset by class:")
    for c,n in df[class_col].value_counts().items(): log(f"    {str(c):<18s}: {n:,}")
if label_col:
    log("\n  Empty-source rows by binary label:")
    for c,n in df[empty_mask][label_col].value_counts().items(): log(f"    {str(c):<12s}: {n:,}")

log("\n  Full source inventory (bucketed):")
buckets = df["source"].apply(bucket)
for b,n in buckets.value_counts().items():
    log(f"    {b:<22s}: {n:,}")

# ── PART 3 (run in both modes, on current benign): COMPOSITION ─
if label_col:
    log("\n[PART 3] BENIGN SOURCE COMPOSITION")
    benign = df[df[label_col]=="benign"].copy()
    benign["bucket"] = benign["source"].apply(bucket)
    log(f"  benign rows: {len(benign):,}")
    comp = benign["bucket"].value_counts()
    log("  by source:")
    for b,n in comp.items():
        sub = benign[benign["bucket"]==b]
        # sample .tr fraction (sample for speed if large)
        s = sub.sample(min(len(sub),20000), random_state=42)
        tr_frac = s["url"].apply(is_tr).mean()*100
        log(f"    {b:<22s}: {n:>8,} ({n/len(benign)*100:5.1f}%)  .tr≈{tr_frac:4.1f}%")
    log("\n  → If most benign is global (low .tr) while malicious is")
    log("    heavily .tr, the is_tr_domain artifact is explained, and")
    log("    source-provenance bias must be disclosed (mitigated by")
    log("    artifact controls + domain-insulated CV, not eliminated).")

# ── PART 2: RELABEL (only in relabel mode) ───────────────────
if MODE == "apply":
    log("\n[PART 2] RELABELLING empty sources as 'aggregated_feed'")
    log("  (preserving 223k real malicious URLs; only the source")
    log("   metadata was missing, not the URL or its label)")
    before_empty = int(empty_mask.sum())
    df.loc[empty_mask, "source"] = "aggregated_feed"
    df.to_csv(CLEAN_OUT, index=False)
    log(f"  relabelled {before_empty:,} rows to source='aggregated_feed'")
    log(f"  row count UNCHANGED: {len(df):,} (no data lost)")
    log(f"  saved → {CLEAN_OUT}")
    log("  ORIGINAL file untouched.")
    log("\n  Updated source inventory:")
    for b,n in df['source'].apply(bucket).value_counts().items():
        log(f"    {b:<22s}: {n:,}")
    log("\n  No re-run of features/Step 3 needed: row set and labels")
    log("  are identical; only a metadata column value changed.")
else:
    log("\n[PART 2] RELABEL — SKIPPED (mode=audit)")
    log("  Findings show empty-source rows are 18% of the corpus and")
    log("  almost entirely MALICIOUS real URLs (label intact, only the")
    log("  source label missing). DO NOT DELETE. Set MODE='relabel' to")
    log("  tag them as source='unknown', preserving all data.")

open(REPORT,"w").write("\n".join(report))
log(f"\n[saved] {REPORT}")

STEP 0D — SOURCE AUDIT  (mode=apply)
  file: dataset_with_folds_v11.csv
  total rows: 1,239,308

[PART 1] EMPTY-SOURCE AUDIT
  rows with empty/missing source: 223,090 (18.00%)

  Empty-source rows by class:
    other_malicious   : 110,214
    phishing          : 94,493
    malware           : 16,633
    scam              : 1,747
    benign            : 3

  For comparison — FULL dataset by class:
    phishing          : 516,621
    other_malicious   : 424,795
    benign            : 184,465
    malware           : 97,262
    scam              : 16,165

  Empty-source rows by binary label:
    malicious   : 223,087
    benign      : 3

  Full source inventory (bucketed):
    usom                  : 812,328
    (empty)               : 223,090
    common_crawl          : 142,353
    umbrella              : 48,384
    urlhaus               : 9,165
    phishtank             : 3,929
    openphish             : 59

[PART 3] BENIGN SOURCE COMPOSITION
  benign rows: 190,740
  by source:
    com

# ============================================================
# STEP 0D — BENIGN-SOURCE ROBUSTNESS CHECK (TUMC)
# ============================================================
# The source audit (Step 0D) showed the benign set is split by
# ccTLD along source lines:
#   Common Crawl benign  = 100% .tr  (Turkish)
#   Umbrella benign      =   0% .tr  (global .com)
# while malicious is USOM-dominated and Turkish-focused. This
# means ccTLD/global cues correlate with benign SOURCE, raising
# the concern that a model separates "global benign slice" from
# "Turkish malicious" rather than benign from malicious.
#
# This script tests whether the headline binary result SURVIVES
# when that confound is removed, under the artifact-controlled
# deployment config (no is_tr_domain, no is_https, inductive).
#
# THREE benign regimes, same malicious set, same features/model:
#   B-FULL : all benign (CC .tr + Umbrella global)   [baseline]
#   B-CCONLY : Common-Crawl benign only (.tr)         [TR-only]
#   B-BAL  : benign down-sampled so its .tr fraction matches
#            the malicious .tr fraction               [balanced]
#
# If binary AUC stays high across all three (esp. B-CCONLY,
# where benign and malicious are BOTH Turkish, so ccTLD cannot
# separate them), the result is NOT a source/ccTLD artifact.
#
# Output: step0e_source_robustness.csv + report
# ============================================================

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import tldextract
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef, f1_score

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11.csv","features_full_v13.csv","features_full_v12_inductive.csv","features_full_v12.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break
OUT = os.path.join(DATA_DIR, "step0e_source_robustness.csv")
REPORT = os.path.join(DATA_DIR, "step0e_source_robustness_report.txt")

# Deployment config: drop artifacts + leaky graph features
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]

_ex = tldextract.TLDExtract()
try: _ex("test.com.tr")
except Exception: _ex = tldextract.TLDExtract(suffix_list_urls=())
def is_tr(u):
    u=str(u); e=_ex(u if u.startswith("http") else "http://"+u)
    return e.suffix.endswith("tr")

report=[]
def log(s): print(s); report.append(s)

log("="*70); log("STEP 0E — BENIGN-SOURCE ROBUSTNESS (binary, inductive, Exp C)"); log("="*70)

df = pd.read_csv(INPUT, low_memory=False)
META={"url","source","tld","label","label_enc","class_final","class_enc","fold","reg_domain"}
FEATURES=[c for c in df.columns if c not in META and c not in LEAKY]
log(f"  rows {len(df):,}, features {len(FEATURES)} (artifacts + leaky removed)")

# Identify benign .tr status and source bucket
df["_is_tr"] = df["url"].apply(is_tr)
src = df["source"].astype(str).str.lower()
df["_cc"] = src.str.contains("common|crawl|cdx", regex=True)
df["_umbrella"] = src.str.contains("umbrella|cisco", regex=True)

benign = df[df["label"]=="benign"]
mal = df[df["label"]=="malicious"]
mal_tr_frac = mal["_is_tr"].mean()
log(f"  malicious .tr fraction: {mal_tr_frac*100:.1f}%")
log(f"  benign .tr fraction:    {benign['_is_tr'].mean()*100:.1f}%")
log(f"  benign from Common Crawl: {benign['_cc'].sum():,}  Umbrella: {benign['_umbrella'].sum():,}")

def run_regime(name, benign_idx):
    """Train/eval binary on selected benign indices + ALL malicious, inductive folds."""
    idx = np.concatenate([benign_idx, mal.index.values])
    sub = df.loc[idx]
    X = sub[FEATURES].fillna(0).values
    y = (sub["label"]=="malicious").astype(int).values
    folds = sub["fold"].values
    proba = np.zeros(len(sub))
    for fid in sorted(set(folds)):
        tr=np.where(folds!=fid)[0]; te=np.where(folds==fid)[0]
        if len(np.unique(y[tr]))<2: continue
        m=HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,
            max_iter=300,random_state=SEED).fit(X[tr],y[tr])
        proba[te]=m.predict_proba(X[te])[:,1]
    auc=roc_auc_score(y,proba)
    mcc=matthews_corrcoef(y,(proba>0.5).astype(int))
    btr=df.loc[benign_idx,"_is_tr"].mean()*100
    log(f"\n  [{name}]")
    log(f"    benign n={len(benign_idx):,} (.tr={btr:.0f}%) + malicious n={len(mal):,}")
    log(f"    binary AUC={auc:.4f}  MCC={mcc:.4f}")
    return {"regime":name,"n_benign":len(benign_idx),"benign_tr_pct":round(btr,1),
            "auc":round(auc,4),"mcc":round(mcc,4)}

rows=[]
# B-FULL: all benign
rows.append(run_regime("B-FULL (all benign)", benign.index.values))
# B-CCONLY: Common Crawl benign only (all .tr) — benign & malicious both Turkish
cc_benign = benign[benign["_cc"]].index.values
if len(cc_benign)>0:
    rows.append(run_regime("B-CCONLY (.tr benign only)", cc_benign))
# B-BAL: down-sample benign so its .tr fraction matches malicious
rng=np.random.RandomState(SEED)
tr_b=benign[benign["_is_tr"]].index.values
gl_b=benign[~benign["_is_tr"]].index.values
# target: benign .tr fraction == mal_tr_frac
# keep all global, sample tr to hit ratio (or vice versa)
if mal_tr_frac>0 and len(gl_b)>0:
    n_tr_target=int(len(gl_b)*mal_tr_frac/(1-mal_tr_frac)) if mal_tr_frac<1 else len(tr_b)
    n_tr_target=min(n_tr_target,len(tr_b))
    bal_idx=np.concatenate([gl_b, rng.choice(tr_b,size=n_tr_target,replace=False)])
    rows.append(run_regime("B-BAL (.tr matched to malicious)", bal_idx))

res=pd.DataFrame(rows); res.to_csv(OUT,index=False)

log("\n"+"="*70); log("INTERPRETATION"); log("="*70)
log("  The decisive regime is B-CCONLY: benign and malicious are")
log("  BOTH Turkish (.tr), so ccTLD/global cues CANNOT separate them.")
log("  If AUC stays high there, the detection is driven by genuine")
log("  benign-vs-malicious structure, NOT by the benign-source/ccTLD")
log("  confound. Report B-CCONLY as the confound-free robustness check.")
log("")
log("  If AUC DROPS sharply in B-CCONLY, the headline number was")
log("  partly source-driven and must be reported with that caveat.")
log("  Either outcome is honest and reportable.")
open(REPORT,"w").write("\n".join(report))
print(f"\n[saved] {OUT}")

STEP 0E — BENIGN-SOURCE ROBUSTNESS (binary, inductive, Exp C)
  rows 1,239,308, features 5 (artifacts + leaky removed)
  malicious .tr fraction: 2.2%
  benign .tr fraction:    74.6%
  benign from Common Crawl: 48,737  Umbrella: 48,384


ValueError: could not convert string to float: 'kept_no_signal'

# ============================================================
# STEP 0E — NORMALISE COMMON CRAWL SOURCE LABELS (TUMC)
# ============================================================
# The source column contains Common Crawl crawl-batch IDs
# (e.g. CC-MAIN-2022-3536, CC-MAIN-2022-975) alongside the
# canonical 'commoncrawl_tr' label. These CC-MAIN-* values are
# Common Crawl's monthly crawl-archive identifiers and denote
# the same source. This script normalises them to a single
# label so source provenance is consistent.
#
# SAFE ORDER:
#   AUDIT  — list every source value matching CC patterns, with
#            counts and their class/label breakdown, so you can
#            confirm they are genuinely Common Crawl benign rows
#            before changing anything. (no write)
#   APPLY  — rewrite matched values to 'commoncrawl_tr', save to
#            a NEW file (original untouched), report before/after.
#
# Pattern matched: 'CC-MAIN' (case-insensitive) anywhere in the
# source string. Review the audit to ensure no false matches.
# ============================================================

In [ ]:
import os, re
import pandas as pd

MODE = "apply"          # "audit" -> inspect; "apply" -> rewrite labels
TARGET_LABEL = "commoncrawl_tr"
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11_srclabeled.csv","features_full_v12.csv",
             "dataset_with_folds_v10.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break
OUT = os.path.join(DATA_DIR, os.path.basename(INPUT).replace(".csv","_srcnorm.csv"))
REPORT = os.path.join(DATA_DIR, "step0f_cc_normalize_report.txt")

# What counts as a Common Crawl batch label
CC_PATTERN = re.compile(r"cc[-_]?main", re.IGNORECASE)

report=[]
def log(s): print(s); report.append(s)

log("="*70); log(f"STEP 0F — NORMALISE COMMON CRAWL SOURCE  (mode={MODE})")
log(f"  file: {os.path.basename(INPUT)}"); log("="*70)

df = pd.read_csv(INPUT, low_memory=False)
if "source" not in df.columns:
    log("  no 'source' column."); open(REPORT,"w").write("\n".join(report)); raise SystemExit

src = df["source"].astype(str)
cc_mask = src.apply(lambda s: bool(CC_PATTERN.search(s)))
already = src.str.lower().eq(TARGET_LABEL.lower())

log(f"\n  total rows: {len(df):,}")
log(f"  rows already '{TARGET_LABEL}': {int(already.sum()):,}")
log(f"  rows matching CC-MAIN pattern: {int(cc_mask.sum()):,}")

# ── AUDIT: show each distinct CC-MAIN value + counts + class ──
log("\n[AUDIT] distinct CC-MAIN-style source values and counts:")
cc_vals = df.loc[cc_mask, "source"].value_counts()
for v,n in cc_vals.items():
    log(f"    {str(v):<28s}: {n:,}")

label_col = "label" if "label" in df.columns else None
class_col = "class_final" if "class_final" in df.columns else ("class" if "class" in df.columns else None)

if label_col:
    log("\n  CC-MAIN rows by binary label (expect ~all benign):")
    for c,n in df.loc[cc_mask, label_col].value_counts().items():
        log(f"    {str(c):<12s}: {n:,}")
    nonben = df.loc[cc_mask & (df[label_col]!="benign")]
    if len(nonben)>0:
        log(f"\n  ⚠ {len(nonben):,} CC-MAIN rows are NOT benign — inspect before applying:")
        for u in nonben["url"].head(10): log(f"      {u}")
        log("  These would be relabelled as a benign Common Crawl source.")
        log("  If they are genuinely malicious, do NOT fold them in blindly.")
    else:
        log("\n  ✓ all CC-MAIN rows are benign — consistent with Common Crawl.")
if class_col:
    log("\n  CC-MAIN rows by class:")
    for c,n in df.loc[cc_mask, class_col].value_counts().items():
        log(f"    {str(c):<18s}: {n:,}")

# sample URLs to eyeball
log("\n  Sample CC-MAIN URLs (verify they look like normal Turkish sites):")
for u in df.loc[cc_mask, "url"].head(8): log(f"    {u}")

# ── APPLY ────────────────────────────────────────────────────
if MODE == "apply":
    log(f"\n[APPLY] rewriting {int(cc_mask.sum()):,} CC-MAIN values to '{TARGET_LABEL}'")
    df.loc[cc_mask, "source"] = TARGET_LABEL
    df.to_csv(OUT, index=False)
    total_cc = int(df["source"].str.lower().eq(TARGET_LABEL.lower()).sum())
    log(f"  row count UNCHANGED: {len(df):,} (only labels rewritten)")
    log(f"  total '{TARGET_LABEL}' now: {total_cc:,}")
    log(f"  saved → {OUT}   (original untouched)")
    log("\n  Updated source value counts:")
    for v,n in df["source"].astype(str).value_counts().head(15).items():
        log(f"    {str(v):<28s}: {n:,}")
    log("\n  No feature/Step-3 re-run needed: only a metadata column changed.")
else:
    log("\n[APPLY] SKIPPED (mode=audit). Review counts + the non-benign")
    log("  check above. If all CC-MAIN rows are benign Common Crawl,")
    log("  set MODE='apply' and rerun to consolidate the label.")

open(REPORT,"w").write("\n".join(report))
log(f"\n[saved] {REPORT}")

Streaming output truncated to the last 5000 lines.
    CC-MAIN-2019-11030          : 1
    CC-MAIN-2022-7495           : 1
    CC-MAIN-2020-97             : 1
    CC-MAIN-2026-9973           : 1
    CC-MAIN-2025-883            : 1
    CC-MAIN-2026-7907           : 1
    CC-MAIN-2026-6203           : 1
    CC-MAIN-2022-7442           : 1
    CC-MAIN-2021-824            : 1
    CC-MAIN-2025-1519           : 1
    CC-MAIN-2017-72             : 1
    CC-MAIN-2021-1169           : 1
    CC-MAIN-2022-6701           : 1
    CC-MAIN-2026-8057           : 1
    CC-MAIN-2022-7490           : 1
    CC-MAIN-2026-11903          : 1
    CC-MAIN-2026-4325           : 1
    CC-MAIN-2026-13052          : 1
    CC-MAIN-2026-13840          : 1
    CC-MAIN-2025-3094           : 1
    CC-MAIN-2022-6680           : 1
    CC-MAIN-2026-15170          : 1
    CC-MAIN-2020-992            : 1
    CC-MAIN-2025-3683           : 1
    CC-MAIN-2026-12877          : 1
    CC-MAIN-2024-1861           : 1
    CC-MAIN-2

# ============================================================
# STEP 0F — KEYWORD-MATCH FIX IMPACT DIAGNOSTIC (TUMC)
# ============================================================
# classification_keywords.py previously matched keywords as raw
# substrings (no token boundary), so short tokens like 'gib',
# 'eft', 'nvi', 'mlm' could fire inside unrelated words ('gibi',
# 'left', 'invitation', 'filmler'). Because that module ALSO
# assigns class labels (reclassify_all), substring false-matches
# could mislabel rows.
#
# The fixed _compile uses token boundaries for short single-word
# keywords. This script QUANTIFIES the impact BEFORE you re-run
# anything: it recomputes labels with OLD vs NEW matching and
# reports how many rows change class, and which.
#
# If the change is tiny (<~0.5%), existing results stand and you
# document the refinement. If larger, re-run feature selection /
# Step 3 on the corrected labels.
#
# Output: step0g_keyword_impact_report.txt (+ changed-rows CSV)
# ============================================================

In [ ]:

import os, re, warnings
import pandas as pd
warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
KW_DIR   = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11_srclabeled_srcnorm.csv","features_full_v12.csv",
             "dataset_with_folds_v10.csv","malicious_v1.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break
REPORT = os.path.join(DATA_DIR, "step0g_keyword_impact_report.txt")
CHANGED = os.path.join(DATA_DIR, "step0g_changed_rows.csv")

import sys; sys.path.insert(0, KW_DIR); sys.path.insert(0, ".")
import classification_keywords as ck

report=[]
def log(s): print(s); report.append(s)

log("="*70); log("STEP 0G — KEYWORD-MATCH FIX IMPACT"); log(f"  file: {os.path.basename(INPUT)}"); log("="*70)

# ---- OLD _compile (substring, the original) for comparison ----
def _compile_OLD(keywords):
    parts=[]
    for kw in keywords:
        if any(c in kw for c in r'\.^$*+?{}[]|()/'):
            parts.append(kw)
        else:
            parts.append(re.escape(kw))
    return "|".join(parts)

PAT_OLD = {c: re.compile(_compile_OLD(k)) for c,k in
           [("phishing",ck.PHISHING_KEYWORDS),("malware",ck.MALWARE_KEYWORDS),
            ("scam",ck.SCAM_KEYWORDS)]}
PAT_NEW = {c: re.compile(ck._compile(k)) for c,k in
           [("phishing",ck.PHISHING_KEYWORDS),("malware",ck.MALWARE_KEYWORDS),
            ("scam",ck.SCAM_KEYWORDS)]}

df = pd.read_csv(INPUT, low_memory=False)
urls = df["url"].astype(str).str.lower().fillna("")
log(f"  rows: {len(df):,}")

def kw_label(urls, PAT):
    p = urls.str.contains(PAT["phishing"].pattern, regex=True, na=False)
    m = urls.str.contains(PAT["malware"].pattern,  regex=True, na=False) & ~p
    s = urls.str.contains(PAT["scam"].pattern,     regex=True, na=False) & ~p & ~m
    lab = pd.Series("none", index=urls.index)
    lab[p]="phishing"; lab[m]="malware"; lab[s]="scam"
    return lab

log("\n  Computing keyword labels with OLD (substring) matching...")
old_lab = kw_label(urls, PAT_OLD)
log("  Computing keyword labels with NEW (token-boundary) matching...")
new_lab = kw_label(urls, PAT_NEW)

# ---- Where do the keyword SIGNALS differ? ----
diff = old_lab != new_lab
log(f"\n  Keyword-signal changes: {int(diff.sum()):,} / {len(df):,} "
    f"({diff.mean()*100:.3f}%)")
if diff.sum():
    log("\n  Signal transitions (old -> new):")
    trans = pd.crosstab(old_lab[diff], new_lab[diff])
    log(trans.to_string())

# ---- Translate to LABEL changes via the actual reclassify logic ----
# Only rows whose source triggers keyword reclassification are affected.
src = df["source"].astype(str)
reclassify_mask = src.isin(ck.RECLASSIFY_ALL_SOURCES)
trusted_mask    = src.isin(ck.KEEP_SOURCE_LABEL)
unknown_mask    = ~reclassify_mask & ~trusted_mask
# rows where the FINAL label could move = signal changed AND source is
# reclassify (always uses kw) or unknown-with-signal
affected = diff & (reclassify_mask | (unknown_mask & (new_lab!="none")) | (unknown_mask & (old_lab!="none")))
log(f"\n  Rows whose FINAL label would change: {int(affected.sum()):,} "
    f"({affected.mean()*100:.3f}%)")
log("  (signal change on a source that uses the keyword signal)")

if affected.sum():
    by_src = src[affected].value_counts().head(8)
    log("\n  Affected rows by source:")
    for s,n in by_src.items(): log(f"    {s:<22s}: {n:,}")
    # save examples for inspection
    ex = df.loc[affected, ["url","source"]].copy()
    ex["old_kw_signal"]=old_lab[affected]; ex["new_kw_signal"]=new_lab[affected]
    ex.to_csv(CHANGED, index=False)
    log(f"\n  Saved {len(ex):,} changed rows -> {os.path.basename(CHANGED)}")
    log("\n  Sample changed URLs (inspect: NEW should be more correct):")
    for _,r in ex.head(15).iterrows():
        log(f"    [{r['old_kw_signal']}->{r['new_kw_signal']}] {r['url'][:70]}")

# ---- Verdict ----
log("\n" + "="*70); log("VERDICT"); log("="*70)
pct = affected.mean()*100
if pct < 0.5:
    log(f"  Label impact is SMALL ({pct:.3f}%). Existing results stand;")
    log("  document the matching refinement as a robustness improvement.")
    log("  Re-running Step 3 is optional (changes are negligible).")
else:
    log(f"  Label impact is MATERIAL ({pct:.3f}%). Re-run reclassify_all")
    log("  with the fixed module, then re-run feature selection (2E) and")
    log("  Step 3 on the corrected labels, and update manuscript counts.")
log("\n  Either way: the fixed token-boundary matching is more correct")
log("  and removes the substring false-match concern from label assignment.")
open(REPORT,"w").write("\n".join(report))
print(f"\n[saved] {REPORT}")

STEP 0G — KEYWORD-MATCH FIX IMPACT
  file: dataset_with_folds_v11_srclabeled_srcnorm.csv
  rows: 1,239,308

  Computing keyword labels with OLD (substring) matching...
  Computing keyword labels with NEW (token-boundary) matching...

  Keyword-signal changes: 4,514 / 1,239,308 (0.364%)

  Signal transitions (old -> new):
col_0     malware  none  scam
row_0                        
phishing      178  4316    20

  Rows whose FINAL label would change: 4,269 (0.344%)
  (signal change on a source that uses the keyword signal)

  Affected rows by source:
    USOM                  : 1,510
    usom                  : 1,238
    aggregated_feed       : 954
    commoncrawl_tr        : 346
    umbrella              : 221

  Saved 4,269 changed rows -> step0g_changed_rows.csv

  Sample changed URLs (inspect: NEW should be more correct):
    [phishing->none] http://nvigelsinparalargelsin.net
    [phishing->none] http://torontochieftains.com/component/option,com_jevents/Itemid,29/da
    [phishing->no

# ============================================================
# STEP 0G — SHORT-KEYWORD COLLISION AUDIT (TUMC)
# ============================================================
# step0g showed 1.35% of labels change under strict token-
# boundary matching, but the changed URLs (e.g. mtv-vergiadesi,
# saglikbakanligiiademerkezim) are REAL Turkish phishing where a
# short keyword ('iade') is legitimately fused into a longer word
# by agglutination. Strict boundaries wrongly drop these.
#
# This audit measures, FOR EACH short single-word keyword
# (< 5 chars), whether substring matching is genuine signal or
# accidental collision, by reporting:
#   - how often it fires (substring) on MALICIOUS vs BENIGN URLs
#   - sample benign URLs it fires on (to spot false collisions)
# A keyword that fires almost only on malicious = genuine Turkish
# morpheme, keep substring. One that fires on benign too = noisy
# collision, needs a boundary.
#
# Output: step0h_short_keyword_audit.txt
# ============================================================

In [ ]:
import os, re, warnings
import pandas as pd
warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
KW_DIR   = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11_srclabeled_srcnorm.csv","features_full_v12.csv","dataset_with_folds_v10.csv"]:
    INPUT=os.path.join(DATA_DIR,cand)
    if os.path.exists(INPUT): break
REPORT=os.path.join(DATA_DIR,"step0h_short_keyword_audit.txt")

import sys; sys.path.insert(0,KW_DIR); sys.path.insert(0,".")
import classification_keywords as ck

report=[]
def log(s): print(s); report.append(s)
log("="*70); log("STEP 0H — SHORT-KEYWORD COLLISION AUDIT"); log("="*70)

df=pd.read_csv(INPUT, low_memory=False)
urls=df["url"].astype(str).str.lower().fillna("")
label_col="label" if "label" in df.columns else None
is_benign = (df[label_col]=="benign").values if label_col else None
log(f"  rows: {len(df):,}  benign: {is_benign.sum():,}  malicious: {(~is_benign).sum():,}")

SHORT_LEN=5
all_short=set()
for lst in [ck.PHISHING_KEYWORDS, ck.MALWARE_KEYWORDS, ck.SCAM_KEYWORDS]:
    for kw in lst:
        if len(kw)<SHORT_LEN and " " not in kw and "-" not in kw \
           and not any(c in kw for c in r'\.^$*+?{}[]|()/'):
            all_short.add(kw)
log(f"\n  Short single-word keywords (<{SHORT_LEN} chars): {len(all_short)}")
log(f"  {sorted(all_short)}\n")

log(f"  {'keyword':<10s} {'mal_hits':>9s} {'ben_hits':>9s} {'ben_rate':>9s}  verdict")
log("  " + "-"*60)
rows=[]
for kw in sorted(all_short):
    # substring match
    hit = urls.str.contains(re.escape(kw), regex=True, na=False).values
    mal_hits = int((hit & ~is_benign).sum())
    ben_hits = int((hit & is_benign).sum())
    total = mal_hits + ben_hits
    ben_rate = ben_hits/total*100 if total else 0.0
    # verdict: low benign rate = genuine signal (keep substring);
    # high benign rate = collision (needs boundary)
    if total < 20:
        verdict="rare (ignore)"
    elif ben_rate < 5:
        verdict="KEEP substring (clean signal)"
    elif ben_rate < 25:
        verdict="borderline — review"
    else:
        verdict="BOUNDARY needed (noisy)"
    log(f"  {kw:<10s} {mal_hits:>9,} {ben_hits:>9,} {ben_rate:>8.1f}%  {verdict}")
    rows.append((kw,mal_hits,ben_hits,ben_rate,verdict))
    # show sample benign collisions for noisy ones
    if ben_rate >= 25 and ben_hits:
        samp = urls[hit & is_benign].head(4).tolist()
        for u in samp: log(f"        benign: {u[:64]}")

log("\n" + "="*70); log("HOW TO USE"); log("="*70)
log("  KEEP substring: genuine Turkish morphemes (fire ~only on")
log("    malicious) — e.g. likely 'iade','odul','onay'. These NEED")
log("    substring matching for agglutination (verg+iade+si).")
log("  BOUNDARY needed: collision-prone tokens firing on benign too")
log("    — e.g. likely 'gib','eft','nvi','mlm'. These get the strict")
log("    boundary so they don't false-match inside ordinary words.")
log("  → Build the allowlist (KEEP) vs boundary-set (NOISY) from this")
log("    table, then I will encode it in _compile so each short")
log("    keyword is handled correctly. This fixes BOTH the English")
log("    false matches AND preserves Turkish agglutinated signal.")
open(REPORT,"w").write("\n".join(report))
print(f"\n[saved] {REPORT}")

STEP 0H — SHORT-KEYWORD COLLISION AUDIT
  rows: 1,239,308  benign: 190,740  malicious: 1,048,568

  Short single-word keywords (<5 chars): 13
  ['2fa', 'eft', 'gib', 'iade', 'iban', 'mhrs', 'mlm', 'n11', 'nvi', 'odul', 'onay', 'sgk', 'worm']

  keyword     mal_hits  ben_hits  ben_rate  verdict
  ------------------------------------------------------------
  2fa              564         8      1.4%  KEEP substring (clean signal)
  eft              532        94     15.0%  borderline — review
  gib            1,349       206     13.2%  borderline — review
  iade          22,878       460      2.0%  KEEP substring (clean signal)
  iban           1,522        22      1.4%  KEEP substring (clean signal)
  mhrs             160         0      0.0%  KEEP substring (clean signal)
  mlm              166         0      0.0%  KEEP substring (clean signal)
  n11              160        16      9.1%  borderline — review
  nvi            2,410       268     10.0%  borderline — review
  odul          

# ============================================================
# STEP 0G2 — VERIFY ALLOWLIST KEYWORD FIX (TUMC)
# ============================================================
# step0g showed the BLUNT boundary fix changed 1.35% of labels,
# but many were REGRESSIONS (real Turkish phishing like
# "vergiadesi" where "iade" is agglutinated). step0h measured
# per-keyword benign-collision rates and we built a data-driven
# ALLOWLIST: genuine morphemes (iade, onay, iban, 2fa, worm,
# mhrs, mlm, odul) keep substring matching; collision-prone
# tokens (sgk, eft, gib, nvi, n11) get strict boundaries.
#
# This script verifies the FINAL allowlist _compile against the
# OLD substring behaviour and confirms the remaining label
# changes are TRUE CORRECTIONS (English/coincidental collisions)
# rather than lost Turkish signal.
#
# Output: step0g2_allowlist_verify_report.txt + changed CSV
# ============================================================

In [ ]:
import os, re, warnings
import pandas as pd
warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
KW_DIR   = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["dataset_with_folds_v11_srclabeled_srcnorm.csv","features_full_v12.csv","dataset_with_folds_v10.csv"]:
    INPUT=os.path.join(DATA_DIR,cand)
    if os.path.exists(INPUT): break
REPORT=os.path.join(DATA_DIR,"step0g2_allowlist_verify_report.txt")
CHANGED=os.path.join(DATA_DIR,"step0g2_changed_rows.csv")

import sys; sys.path.insert(0,KW_DIR); sys.path.insert(0,".")
import importlib, classification_keywords as ck
importlib.reload(ck)   # ensure the NEW allowlist _compile is loaded

report=[]
def log(s): print(s); report.append(s)
log("="*70); log("STEP 0G2 — VERIFY ALLOWLIST KEYWORD FIX"); log("="*70)
log(f"  allowlist KEEP_SUBSTRING_SHORT = {sorted(ck.KEEP_SUBSTRING_SHORT)}")

# OLD substring compile
def _compile_OLD(keywords):
    parts=[]
    for kw in keywords:
        parts.append(kw if any(c in kw for c in r'\.^$*+?{}[]|()/') else re.escape(kw))
    return "|".join(parts)

PAT_OLD={c:re.compile(_compile_OLD(k)) for c,k in
         [("phishing",ck.PHISHING_KEYWORDS),("malware",ck.MALWARE_KEYWORDS),("scam",ck.SCAM_KEYWORDS)]}
PAT_NEW={c:re.compile(ck._compile(k)) for c,k in
         [("phishing",ck.PHISHING_KEYWORDS),("malware",ck.MALWARE_KEYWORDS),("scam",ck.SCAM_KEYWORDS)]}

df=pd.read_csv(INPUT, low_memory=False)
urls=df["url"].astype(str).str.lower().fillna("")
log(f"  rows: {len(df):,}")

def kw_label(urls,PAT):
    p=urls.str.contains(PAT["phishing"].pattern,regex=True,na=False)
    m=urls.str.contains(PAT["malware"].pattern,regex=True,na=False)&~p
    s=urls.str.contains(PAT["scam"].pattern,regex=True,na=False)&~p&~m
    lab=pd.Series("none",index=urls.index); lab[p]="phishing";lab[m]="malware";lab[s]="scam"
    return lab

old_lab=kw_label(urls,PAT_OLD); new_lab=kw_label(urls,PAT_NEW)
diff=old_lab!=new_lab
log(f"\n  Signal changes vs OLD substring: {int(diff.sum()):,} ({diff.mean()*100:.3f}%)")
log("  (these are the COINCIDENTAL collisions the allowlist correctly")
log("   removes — Turkish agglutinated matches like 'iade' are PRESERVED)")
if diff.sum():
    log("\n  Transitions (old substring -> new allowlist):")
    log(pd.crosstab(old_lab[diff],new_lab[diff]).to_string())
    src=df["source"].astype(str)
    reclass=src.isin(ck.RECLASSIFY_ALL_SOURCES)
    trusted=src.isin(ck.KEEP_SOURCE_LABEL)
    unknown=~reclass&~trusted
    affected=diff&(reclass|(unknown&((new_lab!="none")|(old_lab!="none"))))
    log(f"\n  Rows whose FINAL label changes: {int(affected.sum()):,} ({affected.mean()*100:.3f}%)")
    ex=df.loc[affected,["url","source"]].copy()
    ex["old"]=old_lab[affected]; ex["new"]=new_lab[affected]
    ex.to_csv(CHANGED,index=False)
    log("\n  Sample changed URLs (should be English/coincidental, NOT Turkish phish):")
    for _,r in ex.head(20).iterrows():
        log(f"    [{r['old']}->{r['new']}] {r['url'][:68]}")

# Sanity: confirm key Turkish phishing URLs are STILL caught
log("\n  SANITY — these REAL Turkish phishing URLs must still match phishing:")
checks=["http://mtv-vergiadesi.com","http://saglikbakanligiiademerkezim.net",
        "http://aidatiademerkesitr.biz","http://eiadegovtr85.com","http://lmy.de/aidat-iadesi"]
for u in checks:
    hit=bool(PAT_NEW["phishing"].search(u.lower()))
    log(f"    {'✓ MATCH' if hit else '✗ LOST!'}  {u}")

log("\n" + "="*70); log("VERDICT"); log("="*70)
log("  If the changed URLs above are English/coincidental collisions")
log("  (eft-in-left, gib-in-gibi) and the SANITY checks all show ✓,")
log("  the allowlist fix is correct: it removes only false matches and")
log("  preserves Turkish agglutinated signal. Proceed to re-run")
log("  reclassify_all -> 2E -> Step 3 on the corrected labels.")
open(REPORT,"w").write("\n".join(report))
print(f"\n[saved] {REPORT}")

STEP 0G2 — VERIFY ALLOWLIST KEYWORD FIX
  allowlist KEEP_SUBSTRING_SHORT = ['2fa', 'iade', 'iban', 'mhrs', 'mlm', 'odul', 'onay', 'worm']
  rows: 1,239,308

  Signal changes vs OLD substring: 4,514 (0.364%)
  (these are the COINCIDENTAL collisions the allowlist correctly
   removes — Turkish agglutinated matches like 'iade' are PRESERVED)

  Transitions (old substring -> new allowlist):
col_0     malware  none  scam
row_0                        
phishing      178  4316    20

  Rows whose FINAL label changes: 4,269 (0.344%)

  Sample changed URLs (should be English/coincidental, NOT Turkish phish):
    [phishing->none] http://nvigelsinparalargelsin.net
    [phishing->none] http://torontochieftains.com/component/option,com_jevents/Itemid,29/
    [phishing->none] http://nvikolayrandevu.com
    [phishing->malware] http://whatsaapgroupinvit.duckdns.org
    [phishing->none] http://baekwoonvill.com
    [phishing->none] http://heftybtc.com
    [phishing->none] http://e-gibtrafiktahsilat.ml
  

In [ ]:
import pandas as pd, os
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
df = pd.read_csv(os.path.join(DATA_DIR,"dataset_with_folds_v11_srclabeled_srcnorm.csv"), low_memory=False)

# binary label vs 5-class agreement
mal_classes = {"phishing","malware","scam","other_malicious"}
contra_b2m = df[(df["label"]=="benign") & (df["class_final"].isin(mal_classes))]
contra_m2b = df[(df["label"]=="malicious") & (df["class_final"]=="benign")]
print("benign label but malicious class :", len(contra_b2m))
print("malicious label but benign class :", len(contra_m2b))

print("\nbenign-label-malicious-class by source:")
print(contra_b2m["source"].value_counts())
print("\n... by reclassify_reason:")
if "reclassify_reason" in df.columns:
    print(contra_b2m["reclassify_reason"].value_counts())
print("\n... by class_final:")
print(contra_b2m["class_final"].value_counts())
print("\nsample URLs:")
for u in contra_b2m["url"].head(15): print("  ", u)

benign label but malicious class : 6275
malicious label but benign class : 0

benign-label-malicious-class by source:
source
commoncrawl_tr    5819
umbrella           456
Name: count, dtype: int64

... by reclassify_reason:
reclassify_reason
reclassified_unknown_source    6275
Name: count, dtype: int64

... by class_final:
class_final
phishing    4260
scam        1426
malware      589
Name: count, dtype: int64

sample URLs:
   https://mervekolman.av.tr/sgk-isten-cikis-kodlari/
   https://prda.aadg.msidentity.com/
   https://www.samildemir.av.tr/avukat/adware/
   https://www.msenel.av.tr/izmir-avukat/ise-iade-davalari/
   https://buken.av.tr/etiket/idare-mahkemesi-istinaf-basvurusu/
   https://senemyilmazel.av.tr/en/faqs/inheritance-law/
   https://alankodu.tr/upt-fiziki-agini-faturamatik-ptn-ile-guclendirdi/
   https://www.eralp.av.tr/soru-192-sanal-ortamda-islenen-suclar-sozlesmesinin-onaylanmasinin-uygun-bulunduguna-dair-kanun-neler-getiriyor/
   https://memis.av.tr/etiket/papara-dol